In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

!pip install dionysus==2.0.10
import dionysus as d # C++ package with python bindings for persistent homology
import networkx as nx # Network structures
import numpy as np # Numpy arrays and operations
import random # Random sampling for network models
from itertools import combinations, product # For getting different simplices and all combinations of lists

import matplotlib.pyplot as plt # Plotting
import seaborn as sns
import time # Timing simulations
from tqdm.notebook import trange, tqdm # Allows for real-time progress bar of simulations

import gc # Memory management
import pickle # Takes environment variables and saves them as is
import gzip # Allows for compression of saved files
from joblib import Parallel, delayed # Parallelization functions
import multiprocessing # Get number of cpu cores
import gzip

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 57.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for dionysus: filename=dionysus-2.0.10-cp312-cp312-linux_x86_64.whl size=437482 sha256=efab922a2fc69c168abe4974465b8f41247b75139123733b3165bfbf08215916
  Stored in directory: /root/.cache/pip/wheels/11/9f/ca/ae26580795e766a8b6dce75ae4f29163b83b52d7a72efd5d48
Successfully built dionysus


In [ ]:
# First see what is at the top level of the drive mount
print(os.listdir('/content/drive/Othercomputers'))

NameError: name 'os' is not defined

In [ ]:
def build_heatmap(results, field_idx, counts_idx=None):
    """
    Build a 2D heatmap array (len(gammas) x len(alphas)) by averaging
    over iterations for a given output index.
    field_idx: index into ComputeResults output list
               0=Betti0, 1=Betti1, 2=Betti2, 3=Euler, 4=Counts
    counts_idx: if field_idx==4, index into Counts list.
                If the Counts list is shorter than counts_idx+1, returns 0.
    """
    heatmap = np.full((len(gammas), len(alphas)), np.nan)

    for j, alpha in enumerate(alphas):
        for i, gamma in enumerate(gammas):
            alpha_key = round(alpha, 4)
            gamma_key = round(gamma, 4)

            scalars = []
            for iteration in iterations:
                entry = results[alpha_key][gamma_key].get(iteration, None)
                if entry is None:
                    continue
                value = entry[field_idx]
                if counts_idx is not None:
                    if value is not None:
                        # If Counts is shorter than counts_idx+1, use 0
                        scalar = value[counts_idx] if len(value) > counts_idx else 0
                        scalars.append(scalar)
                else:
                    if value is not None:
                        scalars.append(value)

            if scalars:
                heatmap[i, j] = np.mean(scalars)

    return heatmap


def build_normalized_heatmap(results, betti_idx, counts_idx):
    """
    Build a 2D heatmap of Betti[betti_idx] / Counts[counts_idx],
    averaged over iterations. Returns 0 where Counts[counts_idx] == 0.
    betti_idx:  0, 1, or 2 — which Betti number to normalize
    counts_idx: index into Counts list to use as denominator
                (0=nodes, 1=edges, 2=triangles)
    """
    heatmap = np.full((len(gammas), len(alphas)), np.nan)

    for j, alpha in enumerate(alphas):
        for i, gamma in enumerate(gammas):
            alpha_key = round(alpha, 4)
            gamma_key = round(gamma, 4)

            scalars = []
            for iteration in iterations:
                entry = results[alpha_key][gamma_key].get(iteration, None)
                if entry is None:
                    continue
                betti  = entry[betti_idx]   # scalar
                counts = entry[4]           # Counts list

                if betti is None or counts is None:
                    continue

                denom = counts[counts_idx] if len(counts) > counts_idx else 0

                if denom > 0:
                    scalars.append(betti / denom)
                else:
                    scalars.append(0.0)

            if scalars:
                heatmap[i, j] = np.mean(scalars)

    return heatmap


def plot_results_heatmaps(results, n, m, rho,
                          save_path=None, cmap='viridis'):
    """
    Plot heatmaps for:
      - Betti-0, Betti-1, Betti-2
      - Euler characteristic
      - Counts[2] (triangles), Counts[3] (tetrahedra, 0 if absent)
      - Betti-0 / Counts[0], Betti-1 / Counts[1], Betti-2 / Counts[2]
    Each averaged over iterations, alpha on x-axis, gamma on y-axis.
    """
    heatmaps = {
        # Raw Betti numbers
        'Betti-0':                     build_heatmap(results, field_idx=0),
        'Betti-1':                     build_heatmap(results, field_idx=1),
        'Betti-2':                     build_heatmap(results, field_idx=2),
        # Euler characteristic
        'Euler':                       build_heatmap(results, field_idx=3),
        # Simplex counts
        'Counts[2]\n(Triangles)':      build_heatmap(results, field_idx=4,
                                                      counts_idx=2),
        # Tetrahedra — 0 if Counts has length < 4
        'Counts[3]\n(Tetrahedra)':     build_heatmap(results, field_idx=4,
                                                      counts_idx=3),
        # Normalized Betti numbers
        'Betti-0 / Counts[0]\n(Nodes)':     build_normalized_heatmap(results,
                                                betti_idx=0, counts_idx=0),
        'Betti-1 / Counts[1]\n(Edges)':     build_normalized_heatmap(results,
                                                betti_idx=1, counts_idx=1),
        'Betti-2 / Counts[2]\n(Triangles)': build_normalized_heatmap(results,
                                                betti_idx=2, counts_idx=2),
    }

    n_plots = len(heatmaps)
    n_cols  = 3
    n_rows  = int(np.ceil(n_plots / n_cols))

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(6 * n_cols, 5 * n_rows),
        constrained_layout=True
    )
    axes = axes.flatten()

    for ax, (label, heatmap) in zip(axes, heatmaps.items()):
        sns.heatmap(
            heatmap,
            ax          = ax,
            cmap        = cmap,
            xticklabels = tick_interval,
            yticklabels = tick_interval,
            cbar_kws    = {'label': label}
        )
        ax.set_xticks(np.arange(0, len(alphas), tick_interval) + 0.5)
        ax.set_xticklabels(alpha_labels[::tick_interval],
                           rotation=45, ha='right')
        ax.set_yticks(np.arange(0, len(gammas), tick_interval) + 0.5)
        ax.set_yticklabels(gamma_labels[::tick_interval], rotation=0)
        ax.set_xlabel('Alpha (Rewiring Probability)', fontsize=9)
        ax.set_ylabel('Gamma (Triangle Rewiring Probability)', fontsize=9)
        ax.set_title(f'{label}\n(rho={rho}, n={n}, m={m})', fontsize=10)

    for ax in axes[n_plots:]:
        ax.set_visible(False)

    fig.suptitle(
        f'Terminal Complex Properties  |  rho={rho}, n={n}, m={m}',
        fontsize=13,
        fontweight='bold'
    )

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved: {save_path}')
    plt.show()

In [ ]:
tick_interval = 5

n = 5000; m = 10000; alphas = np.linspace(0,1,21); gammas = np.linspace(0,1,21); iterations = range(1,11)
rhos = [0.1,0.2,0.3,0.4]
alpha_labels  = [f'{a:.2f}' for a in alphas]
gamma_labels  = [f'{g:.2f}' for g in gammas]
for rho in tqdm(rhos):
  file_path = f'/content/drive/Othercomputers/My Computer (1)/Dissertation/VoterData/G/Triangle_Graphs_{str(rho).replace('.','_')}.pkl'
  with gzip.open(file_path, 'rb') as f:
    graphs = pickle.load(f)

  results = {
                  round(alpha, 4): {
                      round(gamma, 4): {}
                      for gamma in gammas
                  }
                  for alpha in alphas
              }

  for alpha in tqdm(alphas):
    for gamma in gammas:
      for iteration in iterations:
        alpha_key = round(alpha, 4)
        gamma_key = round(gamma, 4)
        results[alpha_key][gamma_key][iteration] = ComputeResults(graphs[alpha_key][gamma_key][iteration])

  # --- Generate plot ---
  plot_results_heatmaps(
      results   = results,
      n         = n,
      m         = m,
      rho       = rho,
      save_path = (
          f'/content/drive/Othercomputers/My Computer (1)/Dissertation/VoterData/G/Triangle/'
          f'heatmap_complex_n{n}_m{m}_rho{str(rho).replace(".","_")}.png'
      )
  )

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/21 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
def ConstructCliqueComplex(G, k = float('inf')):
    """
    Input a networkx graph G and output the clique simplicial
    complex corresponding to G, where every node, edge and
    clique (up to and including size k) is encoded as simplices.
    """
    # Find all maximal cliques in G which correspond to the maximal simplices
    Cliques = list(nx.find_cliques(G))
    Complex = set(tuple([v]) for v in G.nodes())
    for e in G.edges:
        Complex.add(e)

    # Iterate across each maximal clique, and include each subset of the clique
    # as a simplex in the complex. The set() structure, with sorted(), avoids the issue of
    # including identical simplices multiple times.
    for clique in Cliques:
        numNodes = len(clique)
        # For a clique of n nodes, include the clique itself, its n-1 subsets, n-2 subsets,
        # all the way down to the 2-subsets (edges) and 1-subsets (nodes)
        # We do sorted() to avoid double creating simplices
        clique = sorted(clique)
        for r in range(min(numNodes, k), 2, -1):
            # combinations(S,r) is from itertools and returns iterator corresponding
            # to all r-subsets of S.
            for face in combinations(clique, r):
                Complex.add(face)

    return (Complex)

def nFaces(Complex, n):
    """
    Input a simplicial complex and integer n >= 0 and return all n-simplices (simplices with n+1 nodes)
    """
    # Filter function iterates through the faces of the complex and filters those with n+1 nodes
    return list(filter(lambda face: len(face) == n+1, Complex))

def SimplexCounts(Complex):
    """
    Input a simplical complex and return the counts of simplices of each dimension,
    as well as the dimension of the complex (largest dimension of simplices).
    """
    Counts = [len(nFaces(Complex, 0))]
    Dim = 0
    while True:
        temp = len(nFaces(Complex, Dim + 1))
        if temp > 0:
            Counts.append(temp)
            Dim = Dim + 1
        else:
            break
    return Counts, Dim

def BettiNumbersFromPH(G, p = 2):
  """
  Input a networkx graph G, and prime p to compute the betti
  numbers of the clique complex of G over Z mod p. We restrict
  the complex to include nodes, edges, triangles and tetrahedra.
  This ensures correctness of b0, b1 and b2 with minimal computation
  """
  Complex = ConstructCliqueComplex(G,4)
  Simplices = [(list(simplex), len(simplex)-1) for simplex in Complex]

  # Create filtration of simplicial complexes using Times
  f = d.Filtration(Simplices)
  # Compute persistent homology of filtration
  m = d.homology_persistence(f, p)

  # PH doesn't return betti numbers, it returns persistence pairs
  # Here we loop through pairs, any time between the birth
  # and death of the pair corresponds to the existence of a hole
  Betti = np.zeros((4,3))
  one = np.ones(3)
  dgms = d.init_diagrams(m,f)
  for i, dgm in enumerate(dgms):
    for p in dgm:
      Betti[i][int(p.birth):int(min(p.death,3))] += one[int(p.birth):int(min(p.death,3))]

  return [Betti[0][-1], Betti[1][-1], Betti[2][-1]]

def ComputeResults(G, p = 2):
  """
  Input a networkx graph G, and prime p to compute the betti
  numbers of the clique complex of G over Z mod p. Return the
  exact euler characteristic, as well as Betti numbers b0, b1, b2.
  """
  Complex = ConstructCliqueComplex(G)
  Counts = SimplexCounts(Complex)[0]; Euler = 0
  for i in range(len(Counts)):
    Euler += Counts[i] * (-1)**i

  Simplices = [(list(simplex), len(simplex)-1) for simplex in Complex if len(simplex) <= 4]

  # Create filtration of simplicial complexes using Times
  f = d.Filtration(Simplices)
  # Compute persistent homology of filtration
  m = d.homology_persistence(f, p)

  # PH doesn't return betti numbers, it returns persistence pairs
  # Here we loop through pairs, any time between the birth
  # and death of the pair corresponds to the existence of a hole
  Betti = np.zeros((4,4))
  one = np.ones(4)
  dgms = d.init_diagrams(m,f)
  for i, dgm in enumerate(dgms):
    for p in dgm:
      Betti[i][int(p.birth):int(min(p.death,4))] += one[int(p.birth):int(min(p.death,4))]

  return [Betti[0][-1], Betti[1][-1], Betti[2][-1], Euler, Counts]

In [ ]:
def ZZPH_RewireToRandomVoter(G, rho, alpha, timing = False):
  """
  Input a networkx object G, initial opinion 0 density rho, and
  rewiring probability alpha. Simulate the adaptive network voter
  model on the input graph, where at each step a discordant edge
  is selected uniformly. Then, with probability alpha the edge is rewired
  at random, and with probability 1-alpha one node adopts the opinion
  of its neighbor. Returns the sets of the 0-th, 1-st, 2-nd and 3-rd
  betti numbers, using persistent homology (3-rd betti number is
  truncated betti number), the counts of different dimensional simplices
  at each time step, the set of Euler characteristics, and proportions
  of opinions at each time step.
  """

  if timing == True:
    print("(1/5) Initializing graph, variables and data structures",flush=True)
    start = time.time()

  # Set density of opinion 0 to rho, and 1 to (1 - rho)
  N = len(G.nodes())
  Opinions = np.array([0 if i < int(N*rho) else 1 for i in range(N)])

  # Generate list of all edges where the connected nodes have differing (discordant) opinions
  DiscordantEdges = [edge for edge in G.edges() if Opinions[edge[0]] != Opinions[edge[1]]]

  # Initialize list of opinion proportions
  Proportions = [sum(Opinions)/N]

  # Initialize dictionary which keeps track of when simplices
  # are added and removed, and list of simplex counts
  Cliques = nx.enumerate_all_cliques(G)
  Times = {tuple(clique) : [0] for clique in Cliques}
  SimplexCounts = [ [0] * max(len(c) for c in nx.find_cliques(G)) ]
  for simplex in Times:
    SimplexCounts[0][len(simplex)-1] += 1

  timer = 0

  if timing == True:
    end = time.time()
    print("Initialization complete, time taken : "+str(end - start)+" seconds",flush=True)
    print("(2/5) Beginning network evolution",flush=True)
    start = time.time()

  # Main loop of the model. At each step select a discordant edge at random
  # With probability alpha, the (randomly selected) source node is rewired from the target node to a random
  # node with the same opinion as the source node, and is not already connected to the source node
  # Otherwise (with probability 1 - alpha) the source node adopts the opinion of the target node

  while len(DiscordantEdges) > 0:
    # Copy simplex counts and proportions from previous time step, and update timer
    timer += 1; SimplexCounts.append(SimplexCounts[-1].copy());
    Proportions.append(Proportions[-1])

    # Uniformly select a discordant edge
    edgeChoice = np.random.choice(len(DiscordantEdges))
    edge = DiscordantEdges[edgeChoice]

    # Choose either 0 or 1 to choose which node in the edge is the source and which is the target
    choice = np.random.choice(2)
    source = edge[choice]
    target = edge[(choice + 1) % 2]

    # Rewiring (probability alpha)
    if random.random() < alpha:
      # Removing the edge (source, target) removes simplices, so we find these
      # simplices and remove them. We use a set() for simplices to avoid double
      # adding simplices from maxial cliques, and initialize Simplices to
      # contain (source,target) to reduce need for computation
      Simplices = set([tuple(sorted([source,target]))]); Cliques = nx.find_cliques(G,sorted([source, target])) # <- Returns maximal cliques containing e
      for clique in Cliques:
        numNodes = len(clique)
        # For a clique of n nodes, include the clique itself, its n-1 subsets, n-2 subsets,
        # all the way down to the 3-subsets (triangles). We don't do edges or nodes
        # since we never add/remove nodes, and the only edge we care about has
        # already been added. This reduces computation.
        # We do sorted() to avoid double creating simplices
        clique = sorted(clique)
        for r in range(numNodes, 2, -1):
          # combinations(S,r) is from itertools and returns iterator corresponding
          # to all r-subsets of S.
          for face in combinations(clique, r):
            # Only add simplices containing the edge e
            if (source in face) and (target in face):
              Simplices.add(face)

      # From set of simplices extract simplex counts, and if the simplex
      # is a tetrahedron or smaller add it to Times
      for simplex in Simplices:
        SimplexCounts[-1][len(simplex)-1] -= 1
        if len(simplex) <= 4:
          # Since we are removing the simplex, it must already exist in Times dict
          Times[simplex].append(timer)

      # Remove edge from G
      G.remove_edge(source, target)

      # Since we want to remove edge, which is at index edgeChoice in list,
      # we move the last element of the list into index edgeChoice, and pop
      # the last element, reducing the remove edge complexity from O(n) to O(1).
      DiscordantEdges[edgeChoice] = DiscordantEdges[-1]
      DiscordantEdges.pop()

      while True:
        # Randomly select new target
        newTarget = np.random.choice(N)

        # Check that newTarget is not already connected to source, else draw a different newTarget
        # Since the average degree of a node is low, this should be faster than a deterministic selection
        if (newTarget not in G.neighbors(source)) and (newTarget != source) and (newTarget != target):
          break

      # Add edge (source,newTarget) to G
      G.add_edge(source,newTarget);

      # Find all of the newly added simplices (which must contain source and newTarget by necessity)
      # We use a set() for simplices to avoid double adding simplices from maxial cliques,
      # and initialize Simplices to contain (source,newTarget) to reduce need for computation
      Simplices = set([tuple(sorted([source,newTarget]))]); Cliques = nx.find_cliques(G,sorted([source,newTarget])) # <- Returns maximal cliques containing source and newTarget
      for clique in Cliques:
        numNodes = len(clique)
        # If a newly added simplex is larger than the previously largest simplex,
        # it can only be one larger so be increase the size of simplex counts by one
        if numNodes > len(SimplexCounts[-1]):
          SimplexCounts[-1].append(0)
        # For a clique of n nodes, include the clique itself, its n-1 subsets, n-2 subsets,
        # all the way down to the 3-subsets (triangles). We don't do edges or nodes
        # since we never add/remove nodes, and the only edge we care about has
        # already been added. This reduces computation.
        clique = sorted(clique)
        for r in range(numNodes, 2, -1):
          # combinations(S,r) is from itertools and returns iterator corresponding
          # to all r-subsets of S.
          for face in combinations(clique, r):
            if (source in face) and (newTarget in face):
              Simplices.add(face)

      # From set of simplices extract simplex counts, and if the simplex
      # is a tetrahedron or smaller add it to Times
      for simplex in Simplices:
        SimplexCounts[-1][len(simplex)-1] += 1
        if len(simplex) <= 4:
          # Since we are adding simplices to complex, we don't know if they
          # previously exists and were then removed, i.e. we need to check if
          # they already exist in Times dict
          if simplex in Times:
            Times[simplex].append(timer)
          else:
            Times[simplex] = [timer]

      # Now that the edge has been rewired, check if it is now discordant
      if Opinions[source] != Opinions[newTarget]:
        DiscordantEdges.append( tuple(sorted([source, newTarget])) )

    # Opinion adoption (probability 1 - alpha)
    else:
      # Source node adopts opinion of target node
      Opinions[source] = Opinions[target]

      # Either add 1/n or subtract 1/n to proportion of 1's
      if Opinions[target] == 1:
        Proportions[timer] += 1/N
      else:
        Proportions[timer] -= 1/N

      # Since we have changed the opinion of source node, we need to update whether edges containing
      # source node are discordant or not
      for neighbor in G.neighbors(source):
        # If the opinions of the source and its neighbor differ, then they previously
        # were the same and thus not discordant, must add edge to discordant list
        if Opinions[source] != Opinions[neighbor]:
           DiscordantEdges.append( tuple(sorted([source,neighbor])) )

        # If the opinions of the source and its neighbor are the same, then they
        # previously were discordant and so we must remove the edge from the discordant list
        else:
          DiscordantEdges.remove( tuple(sorted([source,neighbor])) )

  if timing == True:
    end = time.time()
    print("Evolution complete, time taken : "+str(end - start)+" seconds",flush=True)
    print("(3/5) Beginning Zigzag Persistent Homology",flush=True)
    start = time.time()

  # Extract list of every simplex added/removed, and list of times they were
  # added/removed, for input into zigzag persistence.
  simplices = [list(key) for key in Times]; times = [Times[key] for key in Times]

  # Clear out Times, which is massive
  del(Times); del(G); gc.collect()

  # Construct filtration and compute homology
  f = d.Filtration(simplices)
  zz, dgms, cells = d.zigzag_homology_persistence(f, times)

  # Clear out remaining lists which are massive
  del(simplices); del(times); gc.collect()

  if timing == True:
    end = time.time()
    print("Zigzag Persistent Homology complete, time taken : "+str(end - start)+" seconds",flush=True)
    print("(4/5) Beginning Betti number extraction",flush=True)
    start = time.time()

  # PH doesn't return betti numbers, it returns persistence pairs
  # Here we loop through pairs, any time between the birth
  # and death of the pair corresponds to the existence of a hole
  Betti = np.zeros((4,timer+1))
  one = np.ones(timer+1)
  for i, dgm in enumerate(dgms):
    for p in dgm:
      Betti[i][int(p.birth):int(min(p.death,timer+1))] += one[int(p.birth):int(min(p.death,timer+1))]

  if timing == True:
    end = time.time()
    print("Betti number extraction complete, time taken : "+str(end - start)+" seconds")
    print("(5/5) Beginning Euler Characteristic extraction")
    start = time.time()

  # For each time step, compute the euler characteristic as the alternating
  # sum of simplex counts SUM( (-1)^j * num j-simplices )
  Euler = np.zeros(timer+1)
  for i in range(len(SimplexCounts)):
    for j in range(len(SimplexCounts[i])):
      Euler[i] += np.power(-1,j) * SimplexCounts[i][j]

  if timing == True:
    end = time.time()
    print("Euler Characteristic extraction complete, time taken : "+str(end - start)+" seconds")

  return Betti, SimplexCounts, Euler, Proportions

################################################################################

def Betti_RewireToRandomVoter(G, rho, alpha, timing = False):
  """
  Input a networkx object G, initial opinion 0 density rho, and
  rewiring probability alpha. Simulate the adaptive network voter
  model on the input graph, where at each step a discordant edge
  is selected uniformly. Then, with probability alpha the edge is rewired
  at random, and with probability 1-alpha one node adopts the opinion
  of its neighbor. Returns the terminal values of the 0-th, 1-st, 2-nd and 3-rd
  betti numbers, using persistent homology (3-rd betti number is
  truncated betti number), the counts of different dimensional simplices, the Euler characteristic, and proportion
  of opinions at the termination of the model.
  """

  if timing == True:
    print("(1/3) Initializing graph, variables and data structures",flush=True)
    start = time.time()

  # Set density of opinion 0 to rho, and 1 to (1 - rho)
  N = len(G.nodes())
  Opinions = np.array([0 if i < int(N*rho) else 1 for i in range(N)])

  # Generate list of all edges where the connected nodes have differing (discordant) opinions
  DiscordantEdges = [edge for edge in G.edges() if Opinions[edge[0]] != Opinions[edge[1]]]

  timer = 0

  if timing == True:
    end = time.time()
    print("Initialization complete, time taken : "+str(end - start)+" seconds",flush=True)
    print("(2/3) Beginning network evolution",flush=True)
    start = time.time()

  # Main loop of the model. At each step select a discordant edge at random
  # With probability alpha, the (randomly selected) source node is rewired from the target node to a random
  # node with the same opinion as the source node, and is not already connected to the source node
  # Otherwise (with probability 1 - alpha) the source node adopts the opinion of the target node

  while len(DiscordantEdges) > 0:
    # Update timer
    timer += 1;

    # Uniformly select a discordant edge
    edgeChoice = np.random.choice(len(DiscordantEdges))
    edge = DiscordantEdges[edgeChoice]

    # Choose either 0 or 1 to choose which node in the edge is the source and which is the target
    choice = np.random.choice(2)
    source = edge[choice]
    target = edge[(choice + 1) % 2]

    # Rewiring (probability alpha)
    if random.random() < alpha:

      # Remove edge from G
      G.remove_edge(source, target)

      # Since we want to remove edge, which is at index edgeChoice in list,
      # we move the last element of the list into index edgeChoice, and pop
      # the last element, reducing the remove edge complexity from O(n) to O(1).
      DiscordantEdges[edgeChoice] = DiscordantEdges[-1]
      DiscordantEdges.pop()

      while True:
        # Randomly select new target
        newTarget = np.random.choice(N)

        # Check that newTarget is not already connected to source, else draw a different newTarget
        # Since the average degree of a node is low, this should be faster than a deterministic selection
        if (newTarget not in G.neighbors(source)) and (newTarget != source) and (newTarget != target):
          break

      # Add edge (source,newTarget) to G
      G.add_edge(source,newTarget);

      # Now that the edge has been rewired, check if it is now discordant
      if Opinions[source] != Opinions[newTarget]:
        DiscordantEdges.append( tuple(sorted([source, newTarget])) )

    # Opinion adoption (probability 1 - alpha)
    else:
      # Source node adopts opinion of target node
      Opinions[source] = Opinions[target]

      # Since we have changed the opinion of source node, we need to update whether edges containing
      # source node are discordant or not
      for neighbor in G.neighbors(source):
        # If the opinions of the source and its neighbor differ, then they previously
        # were the same and thus not discordant, must add edge to discordant list
        if Opinions[source] != Opinions[neighbor]:
           DiscordantEdges.append( tuple(sorted([source,neighbor])) )

        # If the opinions of the source and its neighbor are the same, then they
        # previously were discordant and so we must remove the edge from the discordant list
        else:
          DiscordantEdges.remove( tuple(sorted([source,neighbor])) )

  if timing == True:
    end = time.time()
    print("Evolution complete, time taken : "+str(end - start)+" seconds",flush=True)
    print("Iterations taken : "+str(timer),flush=True)
    print("(3/3) Computing Final Results",flush=True)
    start = time.time()

  results = ComputeResults(G)
  proportion = sum(Opinions)/N
  proportion = min(proportion, 1 - proportion)
  results.append(proportion)

  if timing == True:
    end = time.time()
    print("Results complete, time taken : "+str(end - start)+" seconds",flush=True)

  # Returns [b0, b1, b2, Euler, Triangle Count, Terminal Minority Proportion]
  return results

In [ ]:
n = 10000; m = 4*n
rho = 0.25; alpha = 0.5
G = nx.gnm_random_graph(n,m)

results = Betti_RewireToRandomVoter(G, rho, alpha, timing = True)

(1/3) Initializing graph, variables and data structures
Initialization complete, time taken : 0.07372856140136719 seconds
(2/3) Beginning network evolution
Evolution complete, time taken : 18979.950709104538 seconds
Iterations take : 26048551
(3/3) Computing Final Results
Results complete, time taken : 1.5066962242126465 seconds


# Sequential Call

In [ ]:
# Set n to be the number of nodes, and m the number of edges
# Set rhos to be the set of densities of opinions
# Set alphas to be the rewiring probability (social selection vs influence)
n = 2000; m = 4000;
rhos = [0.1,0.25,0.5]; alphas = np.linspace(0,1, num = 40)

for alpha in tqdm(alphas):
  for rho in tqdm(rhos):
    G = nx.gnm_random_graph(n,m)
    Betti, SimplexCounts, Euler, Proportions = ZZPH_RewireToRandomVoter(G, rho, alpha)
    Data = [Betti, SimplexCounts, Euler, Proportions]
    output = open('/content/drive/My Drive/Colab Notebooks/RewireRandom/RewireRandom_'+str(n)+'_'+str(m)+'_'+str(rho).replace('.','_')+'_'+str(alpha).replace('.','_')+'.pkl','wb')
    pickle.dump(Data, output); output.close()

# Parallel Call

In [ ]:
def ParallelCall_Random(params):
  n = params[0]; m = params[1]; rho = params[2]; alpha = params[3];
  G = nx.gnm_random_graph(n,m)
  Betti, SimplexCounts, Euler, Proportions = ZZPH_RewireToRandomVoter(G, rho, alpha)
  Data = [Betti, SimplexCounts, Euler, Proportions]
  output = open('/content/drive/My Drive/Colab Notebooks/RewireRandom/RewireRandom_'+str(n)+'_'+str(m)+'_'+str(rho).replace('.','_')+'_'+str(alpha).replace('.','_')+'.pkl','wb')
  pickle.dump(Data, output); output.close()
  return 0

# Set n to be the number of nodes, and m the number of edges
# Set rhos to be the set of densities of opinions
# Set alphas to be the rewiring probability (social selection vs influence)
n = [1000]; m = [2000];
rhos = [0.1,0.25,0.5]; alphas = np.linspace(0,1, num = 100)
params = [n, m, rhos, alphas]
params = [p for p in product(*params)]

# Get number of cpus that can be used
num_cores = multiprocessing.cpu_count()

# Actual call to parallel function
results = Parallel(n_jobs=num_cores)(delayed(ParallelCall_Random)(param) for param in tqdm(params))

  0%|          | 0/300 [00:00<?, ?it/s]

In [ ]:
def ZZPH_RewireToSameVoter(G, rho, alpha, timing = False):
  """
  Input a networkx object G, initial opinion 0 density rho, and
  rewiring probability alpha. Simulate the adaptive network voter
  model on the input graph, where at each step a discordant edge
  is selected uniformly. Then, with probability alpha the edge is rewired
  from a source node to a new target node with the same opinion as the
  source node, and with probability 1-alpha the source node adopts the opinion
  of its neighbor. Returns the sets of the 0-th, 1-st, 2-nd and 3-rd
  betti numbers, using persistent homology (3-rd betti number is
  truncated betti number), the counts of different dimensional simplices
  at each time step, the set of Euler characteristics, and proportions
  of opinions at each time step.
  """

  if timing == True:
    print("(1/5) Initializing graph, variables and data structures",flush=True)
    start = time.time()

  N = len(G.nodes())

  # Set density of opinion 0 to rho, and 1 to 1 - rho
  # Opinions[0] and Opinions[1] contain lists of nodes with opinions 0 and 1 respectively, Opinions[2][i] contains opinion of node i
  Opinions = [ list(range(int(N*rho))), list(range(int(N*rho),N)), np.array([0 if i < int(N*rho) else 1 for i in range(N)]) ]

  # Generate list of all edges where the connected nodes have differing (discordant) opinions
  DiscordantEdges = [edge for edge in G.edges() if Opinions[2][edge[0]] != Opinions[2][edge[1]]]

  # Initialize list for proportions
  Proportions = [sum(Opinions[2])/N]

  # Initialize dictionary which keeps track of when simplices
  # are added and removed, and list of simplex counts
  Cliques = nx.enumerate_all_cliques(G)
  Times = {tuple(clique) : [0] for clique in Cliques}
  SimplexCounts = [ [0] * max(len(c) for c in nx.find_cliques(G)) ]
  for simplex in Times:
    SimplexCounts[0][len(simplex)-1] += 1

  timer = 0

  if timing == True:
    end = time.time()
    print("Initialization complete, time taken : "+str(end - start)+" seconds",flush=True)
    print("(2/5) Beginning network evolution",flush=True)
    start = time.time()

  # Main loop of the model. At each step select a discordant edge at random
  # With probability alpha, the (randomly selected) source node is rewired from the target node to a random
  # node with the same opinion as the source node, and is not already connected to the source node
  # Otherwise (with probability 1 - alpha) the source node adopts the opinion of the target node

  while len(DiscordantEdges) > 0:
    # Copy simplex counts and proportions from previous time step, and update timer
    timer += 1; SimplexCounts.append(SimplexCounts[-1].copy());
    Proportions.append(Proportions[-1])

    # Uniformly select a discordant edge
    edgeChoice = np.random.choice(len(DiscordantEdges))
    edge = DiscordantEdges[edgeChoice]

    # Choose either 0 or 1 to choose which node in the edge is the source and which is the target
    choice = np.random.choice(2)
    source = edge[choice]
    target = edge[(choice + 1) % 2]

    # Rewiring (probability alpha)
    if random.random() < alpha:
      # Check if rewiring is possible (rewiring is not possible if all nodes with the same opinion as the source are already connected to the source)
      if len(Opinions[Opinions[2][source]]) <= len([neighbor for neighbor in G.neighbors(source) if Opinions[2][source] == Opinions[2][neighbor]]) + 1:
        continue

      # Removing the edge (source, target) removes simplices, so we find these
      # simplices and remove them. We use a set() for simplices to avoid double
      # adding simplices from maxial cliques, and initialize Simplices to
      # contain (source,target) to reduce need for computation
      Simplices = set([tuple(sorted([source,target]))]); Cliques = nx.find_cliques(G,sorted([source, target])) # <- Returns maximal cliques containing e
      for clique in Cliques:
        numNodes = len(clique)
        # For a clique of n nodes, include the clique itself, its n-1 subsets, n-2 subsets,
        # all the way down to the 3-subsets (triangles). We don't do edges or nodes
        # since we never add/remove nodes, and the only edge we care about has
        # already been added. This reduces computation.
        # We do sorted() to avoid double creating simplices
        clique = sorted(clique)
        for r in range(numNodes, 2, -1):
          # combinations(S,r) is from itertools and returns iterator corresponding
          # to all r-subsets of S.
          for face in combinations(clique, r):
            # Only add simplices containing the edge e
            if (source in face) and (target in face):
              Simplices.add(face)

      # From set of simplices extract simplex counts, and if the simplex
      # is a tetrahedron or smaller add it to Times
      for simplex in Simplices:
        SimplexCounts[-1][len(simplex)-1] -= 1
        if len(simplex) <= 4:
          # Since we are removing the simplex, it must already exist in Times dict
          Times[simplex].append(timer)

      # Remove edge from G
      G.remove_edge(source, target)

      # Since we want to remove edge, which is at index edgeChoice in list,
      # we move the last element of the list into index edgeChoice, and pop
      # the last element, reducing the remove edge complexity from O(n) to O(1).
      DiscordantEdges[edgeChoice] = DiscordantEdges[-1]
      DiscordantEdges.pop()

      # Select new target node to wire source to from set of nodes with same opinion as source
      while True:
        newTarget = np.random.choice(Opinions[Opinions[2][source]])

        # Check that newTarget is not already connected to source, else draw a different newTarget
        # Since the average degree of a node is low, this should be faster than a deterministic selection
        if (newTarget not in G.neighbors(source)) and (newTarget != source) and (newTarget != target):
          break

      # Add edge (source,newTarget) to G
      G.add_edge(source,newTarget);

      # Find all of the newly added simplices (which must contain source and newTarget by necessity)
      # We use a set() for simplices to avoid double adding simplices from maxial cliques,
      # and initialize Simplices to contain (source,newTarget) to reduce need for computation
      Simplices = set([tuple(sorted([source,newTarget]))]); Cliques = nx.find_cliques(G,sorted([source,newTarget])) # <- Returns maximal cliques containing source and newTarget
      for clique in Cliques:
        numNodes = len(clique)
        # If a newly added simplex is larger than the previously largest simplex,
        # it can only be one larger so be increase the size of simplex counts by one
        if numNodes > len(SimplexCounts[-1]):
          SimplexCounts[-1].append(0)
        # For a clique of n nodes, include the clique itself, its n-1 subsets, n-2 subsets,
        # all the way down to the 3-subsets (triangles). We don't do edges or nodes
        # since we never add/remove nodes, and the only edge we care about has
        # already been added. This reduces computation.
        clique = sorted(clique)
        for r in range(numNodes, 2, -1):
          # combinations(S,r) is from itertools and returns iterator corresponding
          # to all r-subsets of S.
          for face in combinations(clique, r):
            if (source in face) and (newTarget in face):
              Simplices.add(face)

      # From set of simplices extract simplex counts, and if the simplex
      # is a tetrahedron or smaller add it to Times
      for simplex in Simplices:
        SimplexCounts[-1][len(simplex)-1] += 1
        if len(simplex) <= 4:
          # Since we are adding simplices to complex, we don't know if they
          # previously exists and were then removed, i.e. we need to check if
          # they already exist in Times dict
          if simplex in Times:
            Times[simplex].append(timer)
          else:
            Times[simplex] = [timer]

    # Opinion adoption (probability 1 - alpha)
    else:
      # Source node adopts opinion of target node
      Opinions[Opinions[2][source]].remove(source)
      Opinions[2][source] = Opinions[2][target]
      Opinions[Opinions[2][source]].append(source)

      # Either add 1/n or subtract 1/n to proportion of 1's
      if Opinions[2][target] == 1:
        Proportions[timer] += 1/N
      else:
        Proportions[timer] -= 1/N

      # Since we have changed the opinion of source node, we need to update whether edges containing
      # source node are discordant or not
      for neighbor in G.neighbors(source):
        # If the opinions of the source and its neighbor differ, then they previously
        # were the same and thus not discordant, must add edge to discordant list
        if Opinions[2][source] != Opinions[2][neighbor]:
          DiscordantEdges.append( tuple(sorted([source, neighbor])) )

        # If the opinions of the source and its neighbor are the same, then they
        # previously were discordant and so we must remove the edge from the discordant list
        else:
          DiscordantEdges.remove( tuple(sorted([source, neighbor])) )

  if timing == True:
    end = time.time()
    print("Evolution complete, time taken : "+str(end - start)+" seconds",flush=True)
    print("(3/5) Beginning Zigzag Persistent Homology",flush=True)
    start = time.time()

  # Extract list of every simplex added/removed, and list of times they were
  # added/removed, for input into zigzag persistence.
  simplices = [list(key) for key in Times]; times = [Times[key] for key in Times]

  # Clear out Times, which is massive
  del(Times); del(G); gc.collect()

  # Construct filtration and compute homology
  f = d.Filtration(simplices)
  zz, dgms, cells = d.zigzag_homology_persistence(f, times)

  # Clear out remaining lists which are massive
  del(simplices); del(times); gc.collect()

  if timing == True:
    end = time.time()
    print("Zigzag Persistent Homology complete, time taken : "+str(end - start)+" seconds",flush=True)
    print("(4/5) Beginning Betti number extraction",flush=True)
    start = time.time()

  # PH doesn't return betti numbers, it returns persistence pairs
  # Here we loop through pairs, any time between the birth
  # and death of the pair corresponds to the existence of a hole
  Betti = np.zeros((4,timer+1))
  one = np.ones(timer+1)
  for i, dgm in enumerate(dgms):
    for p in dgm:
      Betti[i][int(p.birth):int(min(p.death,timer+1))] += one[int(p.birth):int(min(p.death,timer+1))]

  if timing == True:
    end = time.time()
    print("Betti number extraction complete, time taken : "+str(end - start)+" seconds")
    print("(5/5) Beginning Euler Characteristic extraction")
    start = time.time()

  # For each time step, compute the euler characteristic as the alternating
  # sum of simplex counts SUM( (-1)^j * num j-simplices )
  Euler = np.zeros(timer+1)
  for i in range(len(SimplexCounts)):
    for j in range(len(SimplexCounts[i])):
      Euler[i] += np.power(-1,j) * SimplexCounts[i][j]

  if timing == True:
    end = time.time()
    print("Euler Characteristic extraction complete, time taken : "+str(end - start)+" seconds")

  return Betti, SimplexCounts, Euler, Proportions

# Sequential Call

In [ ]:
# Set n to be the number of nodes, and m the number of edges
# Set rhos to be the set of densities of opinions
# Set alphas to be the rewiring probability (social selection vs influence)
n = 2000; m = 4000;

rhos = [0.1,0.25,0.5]; alphas = np.linspace(0,1, num = 40)
for alpha in tqdm(alphas):
  for rho in tqdm(rhos):
    G = nx.gnm_random_graph(n,m)
    Betti, SimplexCounts, Euler, Proportions = ZZPH_RewireToSameVoter(G, rho, alpha)
    Data = [Betti, SimplexCounts, Euler, Proportions]
    output = open('/content/drive/My Drive/Colab Notebooks/RewireSame/RewireSame_'+str(n)+'_'+str(m)+'_'+str(rho).replace('.','_')+'_'+str(alpha).replace('.','_')+'.pkl','wb')
    pickle.dump(Data, output); output.close()

# Parallel Call

In [ ]:
def ParallelCall_Same(params):
  n = params[0]; m = params[1]; rho = params[2]; alpha = params[3];
  G = nx.gnm_random_graph(n,m)
  Betti, SimplexCounts, Euler, Proportions = ZZPH_RewireToSameVoter(G, rho, alpha)
  Data = [Betti, SimplexCounts, Euler, Proportions]
  output = open('/content/drive/My Drive/Colab Notebooks/RewireSame/RewireSame_'+str(n)+'_'+str(m)+'_'+str(rho).replace('.','_')+'_'+str(alpha).replace('.','_')+'.pkl','wb')
  pickle.dump(Data, output); output.close()
  return 0

# Set n to be the number of nodes, and m the number of edges
# Set rhos to be the set of densities of opinions
# Set alphas to be the rewiring probability (social selection vs influence)
n = [2000]; m = [4000];
rhos = [0.1,0.25,0.5]; alphas = np.linspace(0,1, num = 40)
params = [n, m, rhos, alphas]
params = [p for p in product(*params)]

# Get number of cpus that can be used
num_cores = multiprocessing.cpu_count()

# Actual call to parallel function
results = Parallel(n_jobs=num_cores)(delayed(ParallelCall_Same)(param) for param in tqdm(params))

In [ ]:
def ZZPH_TriangleRewireVoter(G, rho, alpha, gamma, timing = False):
  """
  Input a networkx object G, initial opinion 0 density rho, rewiring probability
  alpha, and triangle closing probability gamma. Simulate the adaptive network
  voter model on the input graph, where at each step a discordant edge
  is selected uniformly. Then, with probability alpha the edge is rewired
  from a source node to a new target node. With probability gamma the new
  target node is a neighbor of a neighbor of the source node, closing the
  triangle, otherwise (with probability 1-gamma) the new target is selected
  randomly. With probability 1-alpha the source node adopts the opinion
  of its neighbor. Returns the sets of the 0-th, 1-st, 2-nd and 3-rd
  betti numbers, using persistent homology (3-rd betti number is
  truncated betti number), the counts of different dimensional simplices
  at each time step, the set of Euler characteristics, and proportions
  of opinions at each time step.
  """

  if timing == True:
    print("(1/5) Initializing graph, variables and data structures",flush=True)
    start = time.time()

  N = len(G.nodes())

  # Set density of opinion 0 to rho, and 1 to 1 - rho
  Opinions = np.array([0 if i < int(N*rho) else 1 for i in range(N)])

  # Generate list of all edges where the connected nodes have differing (discordant) opinions
  DiscordantEdges = [edge for edge in G.edges() if Opinions[2][edge[0]] != Opinions[2][edge[1]]]

  # Initialize list for proportions
  Proportions = [sum(Opinions)/N]

  # Initialize dictionary which keeps track of when simplices
  # are added and removed, and list of simplex counts
  Cliques = nx.enumerate_all_cliques(G)
  Times = {tuple(clique) : [0] for clique in Cliques}
  SimplexCounts = [ [0] * max(len(c) for c in nx.find_cliques(G)) ]
  for simplex in Times:
    SimplexCounts[0][len(simplex)-1] += 1

  timer = 0

  if timing == True:
    end = time.time()
    print("Initialization complete, time taken : "+str(end - start)+" seconds",flush=True)
    print("(2/5) Beginning network evolution",flush=True)
    start = time.time()

  # Main loop of the model. At each step select a discordant edge at random
  # With probability alpha, the (randomly selected) source node is rewired from the target node to
  # neighbor of one of its neighbors with probability gamma, or to a random
  # node with probability 1-gamma. Otherwise (with probability 1 - alpha)
  # the source node adopts the opinion of the target node
  while len(DiscordantEdges) > 0:
    # Copy simplex counts and proportions from previous time step, and update timer
    timer += 1; SimplexCounts.append(SimplexCounts[-1].copy());
    Proportions.append(Proportions[-1])

    # Uniformly select a discordant edge
    edgeChoice = np.random.choice(len(DiscordantEdges))
    edge = DiscordantEdges[edgeChoice]

    # Choose either 0 or 1 to choose which node in the edge is the source and which is the target
    choice = np.random.choice(2)
    source = edge[choice]
    target = edge[(choice + 1) % 2]

    # Rewiring (probability alpha)
    if random.random() < alpha:
      # Removing the edge (source, target) removes simplices, so we find these
      # simplices and remove them. We use a set() for simplices to avoid double
      # adding simplices from maxial cliques, and initialize Simplices to
      # contain (source,target) to reduce need for computation
      Simplices = set([tuple(sorted([source,target]))]); Cliques = nx.find_cliques(G,sorted([source, target])) # <- Returns maximal cliques containing e
      for clique in Cliques:
        numNodes = len(clique)
        # For a clique of n nodes, include the clique itself, its n-1 subsets, n-2 subsets,
        # all the way down to the 3-subsets (triangles). We don't do edges or nodes
        # since we never add/remove nodes, and the only edge we care about has
        # already been added. This reduces computation.
        # We do sorted() to avoid double creating simplices
        clique = sorted(clique)
        for r in range(numNodes, 2, -1):
          # combinations(S,r) is from itertools and returns iterator corresponding
          # to all r-subsets of S.
          for face in combinations(clique, r):
            # Only add simplices containing the edge e
            if (source in face) and (target in face):
              Simplices.add(face)

      # From set of simplices extract simplex counts, and if the simplex
      # is a tetrahedron or smaller add it to Times
      for simplex in Simplices:
        SimplexCounts[-1][len(simplex)-1] -= 1
        if len(simplex) <= 4:
          # Since we are removing the simplex, it must already exist in Times dict
          Times[simplex].append(timer)

      # Remove edge from G
      G.remove_edge(source, target)

      # Since we want to remove edge, which is at index edgeChoice in list,
      # we move the last element of the list into index edgeChoice, and pop
      # the last element, reducing the remove edge complexity from O(n) to O(1).
      DiscordantEdges[edgeChoice] = DiscordantEdges[-1]
      DiscordantEdges.pop()

      # With probability gamma randomly select the new target node to be a neighbor
      # of a neighbor of the source node. If no valid selection exists, default
      # to random selection.
      if random.random() < gamma:
        # Construct the list of neighbors of neighbors of the source node, not including
        # the source nodes neighbors, the source node itself, and any duplicate nodes.
        NeighborsOfNeighbors = list(set([NofN for neighbor in G.neighbors(source) for NofN in G.neighbors(neighbor) if (NofN not in G.neighbors(source) and NofN != source and NofN != target)]))

        # Check that a valud neighbor of neighbor exists
        if len(NeighborsOfNeighbors) > 0:
          newTarget = np.random.choice(NeighborsOfNeighbors)
        # Otherwise resort to random selection
        else:
          while True:
            newTarget = np.random.choice(N)
            # Check that newTarget is not already connected to source, else draw a different newTarget
            # Since the average degree of a node is low, this should be faster than a deterministic selection
            if (newTarget not in G.neighbors(source)) and (newTarget != source) and (newTarget != target):
              break

      # With probability 1-gamma, random selection
      else:
        while True:
          newTarget = np.random.choice(N)
          # Check that newTarget is not already connected to source, else draw a different newTarget
          # Since the average degree of a node is low, this should be faster than a deterministic selection
          if (newTarget not in G.neighbors(source)) and (newTarget != source) and (newTarget != target):
            break

      # Add edge (source,newTarget) to G
      G.add_edge(source,newTarget);

      # Now that the edge has been rewired, check if it is now discordant
      if Opinions[source] != Opinions[newTarget]:
        DiscordantEdges.append( tuple(sorted([source, newTarget])) )

      # Find all of the newly added simplices (which must contain source and newTarget by necessity)
      # We use a set() for simplices to avoid double adding simplices from maxial cliques,
      # and initialize Simplices to contain (source,newTarget) to reduce need for computation
      Simplices = set([tuple(sorted([source,newTarget]))]); Cliques = nx.find_cliques(G,sorted([source,newTarget])) # <- Returns maximal cliques containing source and newTarget
      for clique in Cliques:
        numNodes = len(clique)
        # If a newly added simplex is larger than the previously largest simplex,
        # it can only be one larger so be increase the size of simplex counts by one
        if numNodes > len(SimplexCounts[-1]):
          SimplexCounts[-1].append(0)
        # For a clique of n nodes, include the clique itself, its n-1 subsets, n-2 subsets,
        # all the way down to the 3-subsets (triangles). We don't do edges or nodes
        # since we never add/remove nodes, and the only edge we care about has
        # already been added. This reduces computation.
        clique = sorted(clique)
        for r in range(numNodes, 2, -1):
          # combinations(S,r) is from itertools and returns iterator corresponding
          # to all r-subsets of S.
          for face in combinations(clique, r):
            if (source in face) and (newTarget in face):
              Simplices.add(face)

      # From set of simplices extract simplex counts, and if the simplex
      # is a tetrahedron or smaller add it to Times
      for simplex in Simplices:
        SimplexCounts[-1][len(simplex)-1] += 1
        if len(simplex) <= 4:
          # Since we are adding simplices to complex, we don't know if they
          # previously exists and were then removed, i.e. we need to check if
          # they already exist in Times dict
          if simplex in Times:
            Times[simplex].append(timer)
          else:
            Times[simplex] = [timer]

    # Opinion adoption (probability 1 - alpha)
    else:
      # Source node adopts opinion of target node
      Opinions[source] = Opinions[target]

      # Either add 1/n or subtract 1/n to proportion of 1's
      if Opinions[target] == 1:
        Proportions[timer] += 1/N
      else:
        Proportions[timer] -= 1/N

      # Since we have changed the opinion of source node, we need to update whether edges containing
      # source node are discordant or not
      for neighbor in G.neighbors(source):
        # If the opinions of the source and its neighbor differ, then they previously
        # were the same and thus not discordant, must add edge to discordant list
        if Opinions[source] != Opinions[neighbor]:
          DiscordantEdges.append( tuple(sorted([source, neighbor])) )

        # If the opinions of the source and its neighbor are the same, then they
        # previously were discordant and so we must remove the edge from the discordant list
        else:
          DiscordantEdges.remove( tuple(sorted([source, neighbor])) )

  if timing == True:
    end = time.time()
    print("Evolution complete, time taken : "+str(end - start)+" seconds",flush=True)
    print("(3/5) Beginning Zigzag Persistent Homology",flush=True)
    start = time.time()

  # Extract list of every simplex added/removed, and list of times they were
  # added/removed, for input into zigzag persistence.
  simplices = [list(key) for key in Times]; times = [Times[key] for key in Times]

  # Clear out Times, which is massive
  del(Times); del(G); gc.collect()

  # Construct filtration and compute homology
  f = d.Filtration(simplices)
  zz, dgms, cells = d.zigzag_homology_persistence(f, times)

  # Clear out remaining lists which are massive
  del(simplices); del(times); gc.collect()

  if timing == True:
    end = time.time()
    print("Zigzag Persistent Homology complete, time taken : "+str(end - start)+" seconds",flush=True)
    print("(4/5) Beginning Betti number extraction",flush=True)
    start = time.time()

  # PH doesn't return betti numbers, it returns persistence pairs
  # Here we loop through pairs, any time between the birth
  # and death of the pair corresponds to the existence of a hole
  Betti = np.zeros((4,timer+1))
  one = np.ones(timer+1)
  for i, dgm in enumerate(dgms):
    for p in dgm:
      Betti[i][int(p.birth):int(min(p.death,timer+1))] += one[int(p.birth):int(min(p.death,timer+1))]

  if timing == True:
    end = time.time()
    print("Betti number extraction complete, time taken : "+str(end - start)+" seconds")
    print("(5/5) Beginning Euler Characteristic extraction")
    start = time.time()

  # For each time step, compute the euler characteristic as the alternating
  # sum of simplex counts SUM( (-1)^j * num j-simplices )
  Euler = np.zeros(timer+1)
  for i in range(len(SimplexCounts)):
    for j in range(len(SimplexCounts[i])):
      Euler[i] += np.power(-1,j) * SimplexCounts[i][j]

  if timing == True:
    end = time.time()
    print("Euler Characteristic extraction complete, time taken : "+str(end - start)+" seconds")

  return Betti, SimplexCounts, Euler, Proportions

# Sequential Call

In [ ]:
# Set n to be the number of nodes, and m the number of edges
# Set rhos to be the set of densities of opinions
# Set alphas to be the rewiring probability (social selection vs influence)
# Set gammas to be the probability that given rewiring is occurring, the rewiring
# will be to a neighbor of a neighbor, closing the triangle. Otherwise, the
# rewiring is random
n = 1000; m = 4000;
rhos = [0.25]; alphas = np.linspace(0,1, num = 40); gammas = [.25,.5,.75,1]
Data = {}
for alpha in tqdm(alphas):
  for rho in tqdm(rhos):
    for gamma in tqdm(gammas):
      G = nx.gnm_random_graph(n,m)
      Betti, SimplexCounts, Euler, Proportions = ZZPH_TriangleRewireVoter(G, rho, alpha gamma)
      Data[(alpha,rho,gamma)] = [Betti, SimplexCounts, Euler, Proportions]

output = open('/content/drive/My Drive/Colab Notebooks/RewireTriangle/RewireTriangle_'+str(n)+'_'+str(m)+'.pkl','wb')
pickle.dump(Data, output); output.close()

# Parallel Call

In [ ]:
def ParallelCall_Triangle(params):
  n = params[0]; m = params[1]; rho = params[2]; alpha = params[3]; gamma = params[4];
  G = nx.gnm_random_graph(n,m)
  Betti, SimplexCounts, Euler, Proportions = ZZPH_TriangleRewireVoter(G, rho, alpha, gamma)
  Data = [Betti, SimplexCounts, Euler, Proportions]
  output = open('/content/drive/My Drive/Colab Notebooks/RewireTriangle/RewireTriangle_'+str(n)+'_'+str(m)+'_'+str(rho).replace('.','_')+'_'+str(alpha).replace('.','_')+'_'+str(gamma).replace('.','_')+'.pkl','wb')
  pickle.dump(Data, output); output.close()
  return 0

# Set n to be the number of nodes, and m the number of edges
# Set rhos to be the set of densities of opinions
# Set alphas to be the rewiring probability (social selection vs influence)
# Set gammas to be the probability that given rewiring is occurring, the rewiring
# will be to a neighbor of a neighbor, closing the triangle. Otherwise, the
# rewiring is random
n = [2000]; m = [8000];
rhos = [0.1,0.25,0.5]; alphas = np.linspace(0,1, num = 40); gammas = [.25,.5,.75,1]
params = [n, m, rhos, alphas, gammas]
params = [p for p in product(*params)]

# Get number of cpus that can be used
num_cores = multiprocessing.cpu_count()

# Actual call to parallel function
results = Parallel(n_jobs=num_cores)(delayed(ParallelCall_Triangle)(param) for param in tqdm(params))

def RewireToRandomVoter(G, rho, alpha, timing = False):
    """
    Input a networkx object G, initial opinion 0 density rho, and
    rewiring probability alpha. Simulate the adaptive network voter
    model on the input graph, where at each step a discordant edge
    is selected uniformly. Then, with probability alpha the edge is rewired
    at random, and with probability 1-alpha one node adopts the opinion
    of its neighbor. Returns the proportions of opinions at each time step and
    the terminal graph G.
    """

    if timing == True:
        print("Initializing graph, variables and data structures",flush=True)
        start = time.time()

    # Set density of opinion 0 to rho, and 1 to (1 - rho)
    N = len(G.nodes())
    Opinions = np.array([0 if i < int(N*rho) else 1 for i in range(N)])

    # Generate list of all edges where the connected nodes have differing (discordant) opinions
    DiscordantEdges = [edge for edge in G.edges() if Opinions[edge[0]] != Opinions[edge[1]]]

    # Initialize list of opinion proportions
    Proportions = [sum(Opinions)/N]
    timer = 0

    if timing == True:
        end = time.time()
        print("Initialization complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(Beginning network evolution",flush=True)
        start = time.time()

    # Main loop of the model. At each step select a discordant edge at random
    # With probability alpha, the (randomly selected) source node is rewired from the target node to a random
    # node with the same opinion as the source node, and is not already connected to the source node
    # Otherwise (with probability 1 - alpha) the source node adopts the opinion of the target node

    while len(DiscordantEdges) > 0:
        # Proportions from previous time step, and update timer
        timer += 1; Proportions.append(Proportions[-1])

        # Uniformly select a discordant edge
        edgeChoice = np.random.choice(len(DiscordantEdges))
        edge = DiscordantEdges[edgeChoice]

        # Choose either 0 or 1 to choose which node in the edge is the source and which is the target
        choice = np.random.choice(2)
        source = edge[choice]
        target = edge[(choice + 1) % 2]

        # Rewiring (probability alpha)
        if random.random() < alpha:
            # Remove edge from G
            G.remove_edge(source, target)

            # Since we want to remove edge, which is at index edgeChoice in list,
            # we move the last element of the list into index edgeChoice, and pop
            # the last element, reducing the remove edge complexity from O(n) to O(1).
            DiscordantEdges[edgeChoice] = DiscordantEdges[-1]
            DiscordantEdges.pop()

            while True:
                # Randomly select new target
                newTarget = np.random.choice(N)

                # Check that newTarget is not already connected to source, else draw a different newTarget
                # Since the average degree of a node is low, this should be faster than a deterministic selection
                if (newTarget not in G.neighbors(source)) and (newTarget != source) and (newTarget != target):
                    break

            # Add edge (source,newTarget) to G
            G.add_edge(source,newTarget);

            # Now that the edge has been rewired, check if it is now discordant
            if Opinions[source] != Opinions[newTarget]:
                DiscordantEdges.append( tuple(sorted([source, newTarget])) )

        # Opinion adoption (probability 1 - alpha)
        else:
            # Source node adopts opinion of target node
            Opinions[source] = Opinions[target]

            # Either add 1/n or subtract 1/n to proportion of 1's
            if Opinions[target] == 1:
                Proportions[timer] += 1/N
            else:
                Proportions[timer] -= 1/N

            # Since we have changed the opinion of source node, we need to update whether edges containing
            # source node are discordant or not
            for neighbor in G.neighbors(source):
                # If the opinions of the source and its neighbor differ, then they previously
                # were the same and thus not discordant, must add edge to discordant list
                if Opinions[source] != Opinions[neighbor]:
                    DiscordantEdges.append( tuple(sorted([source,neighbor])) )

                # If the opinions of the source and its neighbor are the same, then they
                # previously were discordant and so we must remove the edge from the discordant list
                else:
                    DiscordantEdges.remove( tuple(sorted([source,neighbor])) )

    if timing == True:
        end = time.time()
        print("Evolution complete, time taken : "+str(end - start)+" seconds",flush=True)

    return Proportions, G

In [ ]:
def ZZPH_RewireToRandomVoter(G, rho, alpha, timing = False):
    """
    Input a networkx object G, initial opinion 0 density rho, and
    rewiring probability alpha. Simulate the adaptive network voter
    model on the input graph, where at each step a discordant edge
    is selected uniformly. Then, with probability alpha the edge is rewired
    at random, and with probability 1-alpha one node adopts the opinion
    of its neighbor. Returns the sets of the 0-th, 1-st, 2-nd and 3-rd
    betti numbers, using persistent homology (3-rd betti number is
    truncated betti number), the counts of different dimensional simplices
    at each time step, the set of Euler characteristics, and proportions
    of opinions at each time step.
    """

    if timing == True:
        print("(1/5) Initializing graph, variables and data structures",flush=True)
        start = time.time()

    # Set density of opinion 0 to rho, and 1 to (1 - rho)
    N = len(G.nodes())
    Opinions = np.array([0 if i < int(N*rho) else 1 for i in range(N)])

    # Generate list of all edges where the connected nodes have differing (discordant) opinions
    DiscordantEdges = [edge for edge in G.edges() if Opinions[edge[0]] != Opinions[edge[1]]]

    # Initialize list of opinion proportions
    Proportions = [sum(Opinions)/N]

    # Initialize dictionary which keeps track of when simplices
    # are added and removed, and list of simplex counts
    Cliques = nx.enumerate_all_cliques(G)
    Times = {tuple(clique) : [0] for clique in Cliques}
    SimplexCounts = [ [0] * max(len(c) for c in nx.find_cliques(G)) ]
    for simplex in Times:
        SimplexCounts[0][len(simplex)-1] += 1

    timer = 0

    if timing == True:
        end = time.time()
        print("Initialization complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(2/5) Beginning network evolution",flush=True)
        start = time.time()

    # Main loop of the model. At each step select a discordant edge at random
    # With probability alpha, the (randomly selected) source node is rewired from the target node to a random
    # node with the same opinion as the source node, and is not already connected to the source node
    # Otherwise (with probability 1 - alpha) the source node adopts the opinion of the target node

    while len(DiscordantEdges) > 0:
        # Copy simplex counts and proportions from previous time step, and update timer
        timer += 1; SimplexCounts.append(SimplexCounts[-1].copy());
        Proportions.append(Proportions[-1])

        # Uniformly select a discordant edge
        edgeChoice = np.random.choice(len(DiscordantEdges))
        edge = DiscordantEdges[edgeChoice]

        # Choose either 0 or 1 to choose which node in the edge is the source and which is the target
        choice = np.random.choice(2)
        source = edge[choice]
        target = edge[(choice + 1) % 2]

        # Rewiring (probability alpha)
        if random.random() < alpha:
          # Removing the edge (source, target) removes simplices, so we find these
          # simplices and remove them. We use a set() for simplices to avoid double
          # adding simplices from maxial cliques, and initialize Simplices to
          # contain (source,target) to reduce need for computation
          Simplices = set([tuple(sorted([source,target]))]); Cliques = nx.find_cliques(G,sorted([source, target])) # <- Returns maximal cliques containing e
          for clique in Cliques:
              numNodes = len(clique)
              # For a clique of n nodes, include the clique itself, its n-1 subsets, n-2 subsets,
              # all the way down to the 3-subsets (triangles). We don't do edges or nodes
              # since we never add/remove nodes, and the only edge we care about has
              # already been added. This reduces computation.
              # We do sorted() to avoid double creating simplices
              clique = sorted(clique)
              for r in range(numNodes, 2, -1):
                  # combinations(S,r) is from itertools and returns iterator corresponding
                  # to all r-subsets of S.
                  for face in combinations(clique, r):
                      # Only add simplices containing the edge e
                      if (source in face) and (target in face):
                          Simplices.add(face)

          # From set of simplices extract simplex counts, and if the simplex
          # is a tetrahedron or smaller add it to Times
          for simplex in Simplices:
              SimplexCounts[-1][len(simplex)-1] -= 1
              if len(simplex) <= 4:
                  # Since we are removing the simplex, it must already exist in Times dict
                  Times[simplex].append(timer)

          # Remove edge from G
          G.remove_edge(source, target)

          # Since we want to remove edge, which is at index edgeChoice in list,
          # we move the last element of the list into index edgeChoice, and pop
          # the last element, reducing the remove edge complexity from O(n) to O(1).
          DiscordantEdges[edgeChoice] = DiscordantEdges[-1]
          DiscordantEdges.pop()

          while True:
              # Randomly select new target
              newTarget = np.random.choice(N)

              # Check that newTarget is not already connected to source, else draw a different newTarget
              # Since the average degree of a node is low, this should be faster than a deterministic selection
              if (newTarget not in G.neighbors(source)) and (newTarget != source) and (newTarget != target):
                  break

          # Add edge (source,newTarget) to G
          G.add_edge(source,newTarget);

          # Find all of the newly added simplices (which must contain source and newTarget by necessity)
          # We use a set() for simplices to avoid double adding simplices from maxial cliques,
          # and initialize Simplices to contain (source,newTarget) to reduce need for computation
          Simplices = set([tuple(sorted([source,newTarget]))]); Cliques = nx.find_cliques(G,sorted([source,newTarget])) # <- Returns maximal cliques containing source and newTarget
          for clique in Cliques:
              numNodes = len(clique)
              # If a newly added simplex is larger than the previously largest simplex,
              # it can only be one larger so be increase the size of simplex counts by one
              if numNodes > len(SimplexCounts[-1]):
                  SimplexCounts[-1].append(0)
              # For a clique of n nodes, include the clique itself, its n-1 subsets, n-2 subsets,
              # all the way down to the 3-subsets (triangles). We don't do edges or nodes
              # since we never add/remove nodes, and the only edge we care about has
              # already been added. This reduces computation.
              clique = sorted(clique)
              for r in range(numNodes, 2, -1):
                  # combinations(S,r) is from itertools and returns iterator corresponding
                  # to all r-subsets of S.
                  for face in combinations(clique, r):
                      if (source in face) and (newTarget in face):
                          Simplices.add(face)

          # From set of simplices extract simplex counts, and if the simplex
          # is a tetrahedron or smaller add it to Times
          for simplex in Simplices:
              SimplexCounts[-1][len(simplex)-1] += 1
              if len(simplex) <= 4:
                  # Since we are adding simplices to complex, we don't know if they
                  # previously exists and were then removed, i.e. we need to check if
                  # they already exist in Times dict
                  if simplex in Times:
                      Times[simplex].append(timer)
                  else:
                      Times[simplex] = [timer]

          # Now that the edge has been rewired, check if it is now discordant
          if Opinions[source] != Opinions[newTarget]:
              DiscordantEdges.append( tuple(sorted([source, newTarget])) )

        # Opinion adoption (probability 1 - alpha)
        else:
            # Source node adopts opinion of target node
            Opinions[source] = Opinions[target]

            # Either add 1/n or subtract 1/n to proportion of 1's
            if Opinions[target] == 1:
                Proportions[timer] += 1/N
            else:
                Proportions[timer] -= 1/N

            # Since we have changed the opinion of source node, we need to update whether edges containing
            # source node are discordant or not
            for neighbor in G.neighbors(source):
                # If the opinions of the source and its neighbor differ, then they previously
                # were the same and thus not discordant, must add edge to discordant list
                if Opinions[source] != Opinions[neighbor]:
                    DiscordantEdges.append( tuple(sorted([source,neighbor])) )

                # If the opinions of the source and its neighbor are the same, then they
                # previously were discordant and so we must remove the edge from the discordant list
                else:
                    DiscordantEdges.remove( tuple(sorted([source,neighbor])) )

    if timing == True:
        end = time.time()
        print("Evolution complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(3/5) Beginning Zigzag Persistent Homology",flush=True)
        start = time.time()

    # Extract list of every simplex added/removed, and list of times they were
    # added/removed, for input into zigzag persistence.
    simplices = [list(key) for key in Times]; times = [Times[key] for key in Times]

    # Clear out Times, which is massive
    del(Times); del(G); gc.collect()

    # Construct filtration and compute homology
    f = d.Filtration(simplices)
    zz, dgms, cells = d.zigzag_homology_persistence(f, times)

    # Clear out remaining lists which are massive
    del(simplices); del(times); gc.collect()

    if timing == True:
        end = time.time()
        print("Zigzag Persistent Homology complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(4/5) Beginning Betti number extraction",flush=True)
        start = time.time()

    # PH doesn't return betti numbers, it returns persistence pairs
    # Here we loop through pairs, any time between the birth
    # and death of the pair corresponds to the existence of a hole
    Betti = np.zeros((4,timer+1))
    one = np.ones(timer+1)
    for i, dgm in enumerate(dgms):
        for p in dgm:
            Betti[i][int(p.birth):int(min(p.death,timer+1))] += one[int(p.birth):int(min(p.death,timer+1))]

    if timing == True:
        end = time.time()
        print("Betti number extraction complete, time taken : "+str(end - start)+" seconds")
        print("(5/5) Beginning Euler Characteristic extraction")
        start = time.time()

    # For each time step, compute the euler characteristic as the alternating
    # sum of simplex counts SUM( (-1)^j * num j-simplices )
    Euler = np.zeros(timer+1)
    for i in range(len(SimplexCounts)):
        for j in range(len(SimplexCounts[i])):
            Euler[i] += np.power(-1,j) * SimplexCounts[i][j]

    if timing == True:
        end = time.time()
        print("Euler Characteristic extraction complete, time taken : "+str(end - start)+" seconds")

    return Betti, SimplexCounts, Euler, Proportions

def ParallelCall_RewireRandom(params):
    G = params[0]; rho = params[1]; alpha = params[2];
    Betti, SimplexCounts, Euler, Proportions = ZZPH_RewireToRandomVoter(G, rho, alpha)
    Data = [Betti, SimplexCounts, Euler, Proportions]
    return Data

In [ ]:
def G_TriangleRewireVoter(G, rho, alpha, gamma, exit_criteria = np.inf, timing = False):
    """
    Input a networkx object G, initial opinion 0 density rho, and
    rewiring probability alpha. Simulate the adaptive network voter
    model on the input graph, where at each step a discordant edge
    is selected uniformly. Then, with probability alpha the edge is rewired
    at random, and with probability 1-alpha one node adopts the opinion
    of its neighbor. Returns the proportions of opinions at each time step and
    the terminal graph G.
    """

    if timing == True:
        print("Initializing graph, variables and data structures",flush=True)
        start = time.time()

    numNodes = len(G.nodes())
    # Set |V|*rho many nodes to opinion 1 and the rest to opinion 0
    Opinions = set(np.random.choice(list(G.nodes()), size=round(rho*numNodes), replace=False))
    Opinions = {v: 1 if v in Opinions else 0 for v in G.nodes()}

    # E will be list of edges and we will keep track of indices of edges in E
    E = list(G.edges())
    N = {v:set() for v in G.nodes()}
    for i, e in enumerate(E):
        for v in e:
            N[v].add(i)
    # Generate list of all edges where the connected nodes have differing (discordant) opinions
    DiscordantEdges = [i for i, edge in enumerate(E) if Opinions[edge[0]] != Opinions[edge[1]]]
    DiscordantIndex = {edge_id: pos for pos, edge_id in enumerate(DiscordantEdges)}

    # Initialize list of opinion proportions
    Proportions = [sum(Opinions.values())/numNodes]
    DiscordantCounts = [len(DiscordantEdges) / len(E)]
    timer = 0

    if timing == True:
        end = time.time()
        print("Initialization complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(Beginning network evolution",flush=True)
        start = time.time()

    # Main loop of the model. At each step select a discordant edge at random
    # With probability alpha, the (randomly selected) source node is rewired from the target node to a random
    # node with the same opinion as the source node, and is not already connected to the source node
    # Otherwise (with probability 1 - alpha) the source node adopts the opinion of the target node

    while len(DiscordantEdges) > 0 and timer < exit_criteria:
        # Timing print
        if timing and timer % 10000 == 0:
            print("Timer : "+str(timer),flush=True)
        # Proportions from previous time step, and update timer
        timer += 1; Proportions.append(Proportions[-1])

        # Uniformly select a discordant edge
        edgeChoice = np.random.randint(len(DiscordantEdges))
        edge_id = DiscordantEdges[edgeChoice]
        edge = E[edge_id]

        # Choose either 0 or 1 to choose which node in the edge is the source and which is the target
        choice = np.random.randint(2)
        source = edge[choice]
        target = edge[(choice + 1) % 2]

        # Rewiring (probability alpha)
        if random.random() < alpha:
            source_neighbors = set(G.neighbors(source))
            target_neighbors = set(G.neighbors(target))

            # Remove edge from G
            G.remove_edge(source, target)
            N[target].remove(edge_id)

            roll = random.random()
            NeighborsOfNeighbors = [None]

            if roll < gamma:
                # Construct the list of neighbors of neighbors of the source node, not including
                # the source nodes neighbors, the source node itself, and any duplicate nodes.
                NeighborsOfNeighbors = list(set(NofN for neighbor in source_neighbors for NofN in G.neighbors(neighbor) if (NofN not in source_neighbors and NofN != source and NofN != target and len(N[NofN]) > 1)))

                # Iterate over valid NofNs checking for valids edges
                while len(NeighborsOfNeighbors) > 0:
                    # Select NofN
                    newTarget = random.choice(NeighborsOfNeighbors)
                    temp = []; counter = 0
                    # Here we check whether the NofN shares 1, or more than 1, neighbor with
                    # the source node. If they share 1, then we need to be careful not to
                    # rewire the shared node as the rewiring would not enforce transitivity
                    for neighbor in G.neighbors(newTarget):
                        if neighbor in source_neighbors:
                            counter += 1
                        else:
                            temp.append(neighbor)
                    # There is at least two common neighbors so we can rewire however we want
                    if counter > 1:
                        cands = list(G.neighbors(newTarget))
                    # There is exactly one common neighbor so we have to be sure not to rewire this neighbor
                    else:
                        cands = temp
                    # Sample and remove invalid cands until a valid node is selected, or no more options exist
                    while cands:
                        idx = random.randrange(len(cands))
                        newSource = cands[idx]
                        if newSource not in target_neighbors and newSource != source and newSource != target:
                            break
                        cands[idx] = cands[-1]
                        cands.pop()
                    # If cands is empty then no valid selection was made
                    if not cands:
                        NeighborsOfNeighbors.remove(newTarget)
                    # Valid selection was made
                    else:
                        # At the end of the gamma branch, after newSource and newTarget are confirmed:
                        choice_id = next(i for i in N[newSource]
                                      if E[i] == tuple(sorted([newSource, newTarget])))
                        break

            # With probability 1-gamma, random selection
            if roll >= gamma or len(NeighborsOfNeighbors) == 0:
                while True:
                    # Randomly select new target
                    choice_id = np.random.randint(len(E))
                    if choice_id == edge_id:
                        continue
                    targetEdge = E[choice_id]
                    choice = np.random.randint(2)
                    newSource = targetEdge[choice]
                    newTarget = targetEdge[(choice + 1) % 2]

                    # Check that newTarget is not already connected to source and newSource is
                    # not already connected to target, else draw a different edge
                    # Since the average degree of a node is low, this should be faster than a deterministic selection
                    if (newTarget not in source_neighbors) and (newTarget != source) and (newTarget != target) and (newSource not in target_neighbors) and (newSource != target) and (newSource != source):
                        break

            # If the edge is now in harmony, then we remove it from DiscordantEdges
            if Opinions[source] == Opinions[newTarget]:
                remove_discordant(edge_id, DiscordantIndex, DiscordantEdges)

            # Remove targeted edge
            G.remove_edge(newSource, newTarget)
            # Update edges incident to the two rewired nodes (dont need to change source nodes)
            N[target].add(choice_id); N[newTarget].remove(choice_id); N[newTarget].add(edge_id)

            # If the targeted edge was discordant and is now in harmony, remove it from DiscordantEdges
            if Opinions[newSource] != Opinions[newTarget] and Opinions[newSource] == Opinions[target]:
                remove_discordant(choice_id, DiscordantIndex, DiscordantEdges)
            # If the targeted edge was harmonious and is now in discord, add it to DiscordantEdges
            elif Opinions[newSource] == Opinions[newTarget] and Opinions[newSource] != Opinions[target]:
                add_discordant(choice_id, DiscordantIndex, DiscordantEdges)
            # Add edges (source, newTarget) and (newSource, target) to G
            G.add_edge(source, newTarget);
            G.add_edge(newSource, target)
            # Update edges in E
            E[edge_id] = tuple(sorted([source, newTarget]))
            E[choice_id] = tuple(sorted([newSource, target]))

        # Opinion adoption (probability 1 - alpha)
        else:
            # Source node adopts opinion of target node
            Opinions[source] = Opinions[target]
            remove_discordant(edge_id, DiscordantIndex, DiscordantEdges)

            # Either add 1/n or subtract 1/n to proportion of 1's
            if Opinions[target] == 1:
                Proportions[timer] += 1/numNodes
            else:
                Proportions[timer] -= 1/numNodes

            # Since we have changed the opinion of source node, we need to update whether edges containing
            # source node are discordant or not
            for Id in N[source]:
                # Already handled this case
                if edge_id == Id:
                    continue
                edge = E[Id]
                # If the opinions of the source and its neighbor differ, then they previously
                # were the same and thus not discordant, must add edge to discordant list
                if Opinions[edge[0]] != Opinions[edge[1]]:
                    add_discordant(Id, DiscordantIndex, DiscordantEdges)
                # If the opinions of the source and its neighbor are the same, then they
                # previously were discordant and so we must remove the edge from the discordant list
                else:
                    remove_discordant(Id, DiscordantIndex, DiscordantEdges)

        DiscordantCounts.append(len(DiscordantEdges) / len(E))

    if timing == True:
        end = time.time()
        print("Evolution complete, time taken : "+str(end - start)+" seconds",flush=True)

    return np.array(Proportions), np.array(DiscordantCounts), G

In [ ]:
def PC_G_RewireTriangle(params):
  """
  Parallel function which is iteratively called,
  looping over a set of parameters, with the current
  iteration of parameters being params.
  """
  # Grab parameter values from params list
  n = params[0]; m = params[1]; rho = params[2]; alpha = params[3]; gamma = params[4]; iteration = params[5];
  # Create filename from params
  ##filename =  'C:/Users/jason/Desktop/Dissertation/VoterData/
  filename =  '/content/drive/My Drive/Colab Notebooks/Voter/Triangle/G_rewire_triangle_'+str(n)+'_'+str(m)+'_'+str(rho).replace('.','_')+'_'+str(alpha).replace('.','_')+'_'+str(gamma).replace('.','_')+'_'+str(iteration)+'.pkl'
  if os.path.isfile(filename):
    return 0

  G = nx.gnm_random_graph(n,m)
  Proportions, DiscordantCounts, G = G_TriangleRewireVoter(G, rho, alpha, gamma)
  data = [Proportions, DiscordantCounts, G]

  # Pickle and save the output
  with gzip.open(filename,'wb') as f:
    pickle.dump(data, f);
  return 0


N = [2000]; M = [2000,4000,6000,8000,10000]; rhos = [0.1,0.2,0.3,0.4]; alphas = np.linspace(0,1,21); iterations = range(1,101);
params = [N, M, rhos, alphas, iterations]
params = [p for p in product(*params)]

# Get number of cpus that can be used
num_cores = multiprocessing.cpu_count()

# Actual call to parallel function
results = Parallel(n_jobs=num_cores)(delayed(PC_G_RewireTriangle)(param) for param in tqdm(params))

In [ ]:
def ZZPH_RewireToSameVoter(G, rho, alpha, timing = False):
    """
    Input a networkx object G, initial opinion 0 density rho, and
    rewiring probability alpha. Simulate the adaptive network voter
    model on the input graph, where at each step a discordant edge
    is selected uniformly. Then, with probability alpha the edge is rewired
    from a source node to a new target node with the same opinion as the
    source node, and with probability 1-alpha the source node adopts the opinion
    of its neighbor. Returns the sets of the 0-th, 1-st, 2-nd and 3-rd
    betti numbers, using persistent homology (3-rd betti number is
    truncated betti number), the counts of different dimensional simplices
    at each time step, the set of Euler characteristics, and proportions
    of opinions at each time step.
    """

    if timing == True:
        print("(1/5) Initializing graph, variables and data structures",flush=True)
        start = time.time()

    N = len(G.nodes())

    # Set density of opinion 0 to rho, and 1 to 1 - rho
    # Opinions[0] and Opinions[1] contain lists of nodes with opinions 0 and 1 respectively, Opinions[2][i] contains opinion of node i
    Opinions = [ list(range(int(N*rho))), list(range(int(N*rho),N)), np.array([0 if i < int(N*rho) else 1 for i in range(N)]) ]

    # Generate list of all edges where the connected nodes have differing (discordant) opinions
    DiscordantEdges = [edge for edge in G.edges() if Opinions[2][edge[0]] != Opinions[2][edge[1]]]

    # Initialize list for proportions
    Proportions = [sum(Opinions[2])/N]

    # Initialize dictionary which keeps track of when simplices
    # are added and removed, and list of simplex counts
    Cliques = nx.enumerate_all_cliques(G)
    Times = {tuple(clique) : [0] for clique in Cliques}
    SimplexCounts = [ [0] * max(len(c) for c in nx.find_cliques(G)) ]
    for simplex in Times:
        SimplexCounts[0][len(simplex)-1] += 1

    timer = 0

    if timing == True:
        end = time.time()
        print("Initialization complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(2/5) Beginning network evolution",flush=True)
        start = time.time()

    # Main loop of the model. At each step select a discordant edge at random
    # With probability alpha, the (randomly selected) source node is rewired from the target node to a random
    # node with the same opinion as the source node, and is not already connected to the source node
    # Otherwise (with probability 1 - alpha) the source node adopts the opinion of the target node

    while len(DiscordantEdges) > 0:
        # Copy simplex counts and proportions from previous time step, and update timer
        timer += 1; SimplexCounts.append(SimplexCounts[-1].copy());
        Proportions.append(Proportions[-1])

        # Uniformly select a discordant edge
        edgeChoice = np.random.choice(len(DiscordantEdges))
        edge = DiscordantEdges[edgeChoice]

        # Choose either 0 or 1 to choose which node in the edge is the source and which is the target
        choice = np.random.choice(2)
        source = edge[choice]
        target = edge[(choice + 1) % 2]

        # Rewiring (probability alpha)
        if random.random() < alpha:
          # Check if rewiring is possible (rewiring is not possible if all nodes with the same opinion as the source are already connected to the source)
          if len(Opinions[Opinions[2][source]]) <= len([neighbor for neighbor in G.neighbors(source) if Opinions[2][source] == Opinions[2][neighbor]]) + 1:
              continue

          # Removing the edge (source, target) removes simplices, so we find these
          # simplices and remove them. We use a set() for simplices to avoid double
          # adding simplices from maxial cliques, and initialize Simplices to
          # contain (source,target) to reduce need for computation
          Simplices = set([tuple(sorted([source,target]))]); Cliques = nx.find_cliques(G,sorted([source, target])) # <- Returns maximal cliques containing e
          for clique in Cliques:
              numNodes = len(clique)
              # For a clique of n nodes, include the clique itself, its n-1 subsets, n-2 subsets,
              # all the way down to the 3-subsets (triangles). We don't do edges or nodes
              # since we never add/remove nodes, and the only edge we care about has
              # already been added. This reduces computation.
              # We do sorted() to avoid double creating simplices
              clique = sorted(clique)
              for r in range(numNodes, 2, -1):
                  # combinations(S,r) is from itertools and returns iterator corresponding
                  # to all r-subsets of S.
                  for face in combinations(clique, r):
                      # Only add simplices containing the edge e
                      if (source in face) and (target in face):
                          Simplices.add(face)

          # From set of simplices extract simplex counts, and if the simplex
          # is a tetrahedron or smaller add it to Times
          for simplex in Simplices:
              SimplexCounts[-1][len(simplex)-1] -= 1
              if len(simplex) <= 4:
                  # Since we are removing the simplex, it must already exist in Times dict
                  Times[simplex].append(timer)

          # Remove edge from G
          G.remove_edge(source, target)

          # Since we want to remove edge, which is at index edgeChoice in list,
          # we move the last element of the list into index edgeChoice, and pop
          # the last element, reducing the remove edge complexity from O(n) to O(1).
          DiscordantEdges[edgeChoice] = DiscordantEdges[-1]
          DiscordantEdges.pop()

          # Select new target node to wire source to from set of nodes with same opinion as source
          while True:
              newTarget = np.random.choice(Opinions[Opinions[2][source]])

              # Check that newTarget is not already connected to source, else draw a different newTarget
              # Since the average degree of a node is low, this should be faster than a deterministic selection
              if (newTarget not in G.neighbors(source)) and (newTarget != source) and (newTarget != target):
                  break

          # Add edge (source,newTarget) to G
          G.add_edge(source,newTarget);

          # Find all of the newly added simplices (which must contain source and newTarget by necessity)
          # We use a set() for simplices to avoid double adding simplices from maxial cliques,
          # and initialize Simplices to contain (source,newTarget) to reduce need for computation
          Simplices = set([tuple(sorted([source,newTarget]))]); Cliques = nx.find_cliques(G,sorted([source,newTarget])) # <- Returns maximal cliques containing source and newTarget
          for clique in Cliques:
              numNodes = len(clique)
              # If a newly added simplex is larger than the previously largest simplex,
              # it can only be one larger so be increase the size of simplex counts by one
              if numNodes > len(SimplexCounts[-1]):
                  SimplexCounts[-1].append(0)
              # For a clique of n nodes, include the clique itself, its n-1 subsets, n-2 subsets,
              # all the way down to the 3-subsets (triangles). We don't do edges or nodes
              # since we never add/remove nodes, and the only edge we care about has
              # already been added. This reduces computation.
              clique = sorted(clique)
              for r in range(numNodes, 2, -1):
                  # combinations(S,r) is from itertools and returns iterator corresponding
                  # to all r-subsets of S.
                  for face in combinations(clique, r):
                      if (source in face) and (newTarget in face):
                          Simplices.add(face)

          # From set of simplices extract simplex counts, and if the simplex
          # is a tetrahedron or smaller add it to Times
          for simplex in Simplices:
              SimplexCounts[-1][len(simplex)-1] += 1
              if len(simplex) <= 4:
                  # Since we are adding simplices to complex, we don't know if they
                  # previously exists and were then removed, i.e. we need to check if
                  # they already exist in Times dict
                  if simplex in Times:
                      Times[simplex].append(timer)
                  else:
                      Times[simplex] = [timer]

        # Opinion adoption (probability 1 - alpha)
        else:
            # Source node adopts opinion of target node
            Opinions[Opinions[2][source]].remove(source)
            Opinions[2][source] = Opinions[2][target]
            Opinions[Opinions[2][source]].append(source)

            # Either add 1/n or subtract 1/n to proportion of 1's
            if Opinions[2][target] == 1:
                Proportions[timer] += 1/N
            else:
                Proportions[timer] -= 1/N

            # Since we have changed the opinion of source node, we need to update whether edges containing
            # source node are discordant or not
            for neighbor in G.neighbors(source):
                # If the opinions of the source and its neighbor differ, then they previously
                # were the same and thus not discordant, must add edge to discordant list
                if Opinions[2][source] != Opinions[2][neighbor]:
                    DiscordantEdges.append( tuple(sorted([source, neighbor])) )

                # If the opinions of the source and its neighbor are the same, then they
                # previously were discordant and so we must remove the edge from the discordant list
                else:
                    DiscordantEdges.remove( tuple(sorted([source, neighbor])) )

    if timing == True:
        end = time.time()
        print("Evolution complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(3/5) Beginning Zigzag Persistent Homology",flush=True)
        start = time.time()

    # Extract list of every simplex added/removed, and list of times they were
    # added/removed, for input into zigzag persistence.
    simplices = [list(key) for key in Times]; times = [Times[key] for key in Times]

    # Clear out Times, which is massive
    del(Times); del(G); gc.collect()

    # Construct filtration and compute homology
    f = d.Filtration(simplices)
    zz, dgms, cells = d.zigzag_homology_persistence(f, times)

    # Clear out remaining lists which are massive
    del(simplices); del(times); gc.collect()

    if timing == True:
        end = time.time()
        print("Zigzag Persistent Homology complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(4/5) Beginning Betti number extraction",flush=True)
        start = time.time()

    # PH doesn't return betti numbers, it returns persistence pairs
    # Here we loop through pairs, any time between the birth
    # and death of the pair corresponds to the existence of a hole
    Betti = np.zeros((4,timer+1))
    one = np.ones(timer+1)
    for i, dgm in enumerate(dgms):
        for p in dgm:
            Betti[i][int(p.birth):int(min(p.death,timer+1))] += one[int(p.birth):int(min(p.death,timer+1))]

    if timing == True:
      end = time.time()
      print("Betti number extraction complete, time taken : "+str(end - start)+" seconds")
      print("(5/5) Beginning Euler Characteristic extraction")
      start = time.time()

    # For each time step, compute the euler characteristic as the alternating
    # sum of simplex counts SUM( (-1)^j * num j-simplices )
    Euler = np.zeros(timer+1)
    for i in range(len(SimplexCounts)):
        for j in range(len(SimplexCounts[i])):
            Euler[i] += np.power(-1,j) * SimplexCounts[i][j]

    if timing == True:
        end = time.time()
        print("Euler Characteristic extraction complete, time taken : "+str(end - start)+" seconds")

    return Betti, SimplexCounts, Euler, Proportions

def ParallelCall_RewireSame(params):
    """
    """
    G = params[0]; rho = params[1]; alpha = params[2];
    Betti, SimplexCounts, Euler, Proportions = ZZPH_RewireToSameVoter(G, rho, alpha)
    Data = [Betti, SimplexCounts, Euler, Proportions]
    return 0

def Parallel_RewireSame(G, rhos, alphas):
    """
    """
    # Set rhos to be the set of initial densities of opinions
    # Set alphas to be the rewiring probability (social selection vs influence)
    params = [[G], rhos, alphas]
    params = [p for p in product(*params)]

    # Get number of cpus that can be used
    num_cores = multiprocessing.cpu_count()

    # Actual call to parallel function
    results = Parallel(n_jobs=num_cores)(delayed(ParallelCall_RewireSame)(param) for param in tqdm(params))


In [ ]:
def ZZPH_TriangleRewireVoter(G, rho, alpha, gamma, timing = False):
    """
    Input a networkx object G, initial opinion 0 density rho, rewiring probability
    alpha, and triangle closing probability gamma. Simulate the adaptive network
    voter model on the input graph, where at each step a discordant edge
    is selected uniformly. Then, with probability alpha the edge is rewired
    from a source node to a new target node. With probability gamma the new
    target node is a neighbor of a neighbor of the source node, closing the
    triangle, otherwise (with probability 1-gamma) the new target is selected
    randomly. With probability 1-alpha the source node adopts the opinion
    of its neighbor. Returns the sets of the 0-th, 1-st, 2-nd and 3-rd
    betti numbers, using persistent homology (3-rd betti number is
    truncated betti number), the counts of different dimensional simplices
    at each time step, the set of Euler characteristics, and proportions
    of opinions at each time step.
    """

    if timing == True:
        print("(1/5) Initializing graph, variables and data structures",flush=True)
        start = time.time()

    N = len(G.nodes())

    # Set density of opinion 0 to rho, and 1 to 1 - rho
    Opinions = np.array([0 if i < int(N*rho) else 1 for i in range(N)])

    # Generate list of all edges where the connected nodes have differing (discordant) opinions
    DiscordantEdges = [edge for edge in G.edges() if Opinions[edge[0]] != Opinions[edge[1]]]

    # Initialize list for proportions
    Proportions = [sum(Opinions)/N]

    # Initialize dictionary which keeps track of when simplices
    # are added and removed, and list of simplex counts
    Cliques = nx.enumerate_all_cliques(G)
    Times = {tuple(clique) : [0] for clique in Cliques}
    SimplexCounts = [ [0] * max(len(c) for c in nx.find_cliques(G)) ]
    for simplex in Times:
      SimplexCounts[0][len(simplex)-1] += 1

    timer = 0

    if timing == True:
        end = time.time()
        print("Initialization complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(2/5) Beginning network evolution",flush=True)
        start = time.time()

    # Main loop of the model. At each step select a discordant edge at random
    # With probability alpha, the (randomly selected) source node is rewired from the target node to
    # neighbor of one of its neighbors with probability gamma, or to a random
    # node with probability 1-gamma. Otherwise (with probability 1 - alpha)
    # the source node adopts the opinion of the target node
    while len(DiscordantEdges) > 0:
        # Copy simplex counts and proportions from previous time step, and update timer
        timer += 1; SimplexCounts.append(SimplexCounts[-1].copy());
        Proportions.append(Proportions[-1])

        # Uniformly select a discordant edge
        edgeChoice = np.random.choice(len(DiscordantEdges))
        edge = DiscordantEdges[edgeChoice]

        # Choose either 0 or 1 to choose which node in the edge is the source and which is the target
        choice = np.random.choice(2)
        source = edge[choice]
        target = edge[(choice + 1) % 2]

        # Rewiring (probability alpha)
        if random.random() < alpha:
            # Removing the edge (source, target) removes simplices, so we find these
            # simplices and remove them. We use a set() for simplices to avoid double
            # adding simplices from maxial cliques, and initialize Simplices to
            # contain (source,target) to reduce need for computation
            Simplices = set([tuple(sorted([source,target]))]); Cliques = nx.find_cliques(G,sorted([source, target])) # <- Returns maximal cliques containing e
            for clique in Cliques:
                numNodes = len(clique)
                # For a clique of n nodes, include the clique itself, its n-1 subsets, n-2 subsets,
                # all the way down to the 3-subsets (triangles). We don't do edges or nodes
                # since we never add/remove nodes, and the only edge we care about has
                # already been added. This reduces computation.
                # We do sorted() to avoid double creating simplices
                clique = sorted(clique)
                for r in range(numNodes, 2, -1):
                    # combinations(S,r) is from itertools and returns iterator corresponding
                    # to all r-subsets of S.
                    for face in combinations(clique, r):
                        # Only add simplices containing the edge e
                        if (source in face) and (target in face):
                            Simplices.add(face)

            # From set of simplices extract simplex counts, and if the simplex
            # is a tetrahedron or smaller add it to Times
            for simplex in Simplices:
                SimplexCounts[-1][len(simplex)-1] -= 1
                if len(simplex) <= 4:
                    # Since we are removing the simplex, it must already exist in Times dict
                    Times[simplex].append(timer)

            # Remove edge from G
            G.remove_edge(source, target)

            # Since we want to remove edge, which is at index edgeChoice in list,
            # we move the last element of the list into index edgeChoice, and pop
            # the last element, reducing the remove edge complexity from O(n) to O(1).
            DiscordantEdges[edgeChoice] = DiscordantEdges[-1]
            DiscordantEdges.pop()

            # With probability gamma randomly select the new target node to be a neighbor
            # of a neighbor of the source node. If no valid selection exists, default
            # to random selection.
            if random.random() < gamma:
                # Construct the list of neighbors of neighbors of the source node, not including
                # the source nodes neighbors, the source node itself, and any duplicate nodes.
                NeighborsOfNeighbors = list(set([NofN for neighbor in G.neighbors(source) for NofN in G.neighbors(neighbor) if (NofN not in G.neighbors(source) and NofN != source and NofN != target)]))

                # Check that a valid neighbor of neighbor exists
                if len(NeighborsOfNeighbors) > 0:
                    newTarget = np.random.choice(NeighborsOfNeighbors)
                # Otherwise resort to random selection
                else:
                    while True:
                        newTarget = np.random.choice(N)
                        # Check that newTarget is not already connected to source, else draw a different newTarget
                        # Since the average degree of a node is low, this should be faster than a deterministic selection
                        if (newTarget not in G.neighbors(source)) and (newTarget != source) and (newTarget != target):
                            break

            # With probability 1-gamma, random selection
            else:
                while True:
                    newTarget = np.random.choice(N)
                    # Check that newTarget is not already connected to source, else draw a different newTarget
                    # Since the average degree of a node is low, this should be faster than a deterministic selection
                    if (newTarget not in G.neighbors(source)) and (newTarget != source) and (newTarget != target):
                        break

            # Add edge (source,newTarget) to G
            G.add_edge(source,newTarget);

            # Now that the edge has been rewired, check if it is now discordant
            if Opinions[source] != Opinions[newTarget]:
                DiscordantEdges.append( tuple(sorted([source, newTarget])) )

            # Find all of the newly added simplices (which must contain source and newTarget by necessity)
            # We use a set() for simplices to avoid double adding simplices from maxial cliques,
            # and initialize Simplices to contain (source,newTarget) to reduce need for computation
            Simplices = set([tuple(sorted([source,newTarget]))]); Cliques = nx.find_cliques(G,sorted([source,newTarget])) # <- Returns maximal cliques containing source and newTarget
            for clique in Cliques:
                numNodes = len(clique)
                # If a newly added simplex is larger than the previously largest simplex,
                # it can only be one larger so be increase the size of simplex counts by one
                if numNodes > len(SimplexCounts[-1]):
                    SimplexCounts[-1].append(0)
                # For a clique of n nodes, include the clique itself, its n-1 subsets, n-2 subsets,
                # all the way down to the 3-subsets (triangles). We don't do edges or nodes
                # since we never add/remove nodes, and the only edge we care about has
                # already been added. This reduces computation.
                clique = sorted(clique)
                for r in range(numNodes, 2, -1):
                    # combinations(S,r) is from itertools and returns iterator corresponding
                    # to all r-subsets of S.
                    for face in combinations(clique, r):
                        if (source in face) and (newTarget in face):
                            Simplices.add(face)

            # From set of simplices extract simplex counts, and if the simplex
            # is a tetrahedron or smaller add it to Times
            for simplex in Simplices:
                SimplexCounts[-1][len(simplex)-1] += 1
                if len(simplex) <= 4:
                    # Since we are adding simplices to complex, we don't know if they
                    # previously exists and were then removed, i.e. we need to check if
                    # they already exist in Times dict
                    if simplex in Times:
                        Times[simplex].append(timer)
                    else:
                        Times[simplex] = [timer]

        # Opinion adoption (probability 1 - alpha)
        else:
            # Source node adopts opinion of target node
            Opinions[source] = Opinions[target]

            # Either add 1/n or subtract 1/n to proportion of 1's
            if Opinions[target] == 1:
                Proportions[timer] += 1/N
            else:
                Proportions[timer] -= 1/N

            # Since we have changed the opinion of source node, we need to update whether edges containing
            # source node are discordant or not
            for neighbor in G.neighbors(source):
                # If the opinions of the source and its neighbor differ, then they previously
                # were the same and thus not discordant, must add edge to discordant list
                if Opinions[source] != Opinions[neighbor]:
                    DiscordantEdges.append( tuple(sorted([source, neighbor])) )

                # If the opinions of the source and its neighbor are the same, then they
                # previously were discordant and so we must remove the edge from the discordant list
                else:
                    DiscordantEdges.remove( tuple(sorted([source, neighbor])) )

    if timing == True:
        end = time.time()
        print("Evolution complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(3/5) Beginning Zigzag Persistent Homology",flush=True)
        start = time.time()

    # Extract list of every simplex added/removed, and list of times they were
    # added/removed, for input into zigzag persistence.
    simplices = [list(key) for key in Times]; times = [Times[key] for key in Times]

    # Clear out Times, which is massive
    del(Times); del(G); gc.collect()

    # Construct filtration and compute homology
    f = d.Filtration(simplices)
    zz, dgms, cells = d.zigzag_homology_persistence(f, times)

    # Clear out remaining lists which are massive
    del(simplices); del(times); gc.collect()

    if timing == True:
        end = time.time()
        print("Zigzag Persistent Homology complete, time taken : "+str(end - start)+" seconds",flush=True)
        print("(4/5) Beginning Betti number extraction",flush=True)
        start = time.time()

    # PH doesn't return betti numbers, it returns persistence pairs
    # Here we loop through pairs, any time between the birth
    # and death of the pair corresponds to the existence of a hole
    Betti = np.zeros((4,timer+1))
    one = np.ones(timer+1)
    for i, dgm in enumerate(dgms):
        for p in dgm:
            Betti[i][int(p.birth):int(min(p.death,timer+1))] += one[int(p.birth):int(min(p.death,timer+1))]

    if timing == True:
      end = time.time()
      print("Betti number extraction complete, time taken : "+str(end - start)+" seconds")
      print("(5/5) Beginning Euler Characteristic extraction")
      start = time.time()

    # For each time step, compute the euler characteristic as the alternating
    # sum of simplex counts SUM( (-1)^j * num j-simplices )
    Euler = np.zeros(timer+1)
    for i in range(len(SimplexCounts)):
        for j in range(len(SimplexCounts[i])):
            Euler[i] += np.power(-1,j) * SimplexCounts[i][j]

    if timing == True:
        end = time.time()
        print("Euler Characteristic extraction complete, time taken : "+str(end - start)+" seconds")

    return Betti, SimplexCounts, Euler, Proportions


def ParallelCall_RewireTriangle(params):
    """
    """
    G = params[0]; rho = params[1]; alpha = params[2]; gamma = params[3]
    Betti, SimplexCounts, Euler, Proportions = ZZPH_TriangleRewireVoter(G, rho, alpha, gamma)
    Data = [Betti, SimplexCounts, Euler, Proportions]
    return 0

def Parallel_RewireTriangle(G, rhos, alphas, gammas):
    """
    """
    # Set rhos to be the set of initial densities of opinions
    # Set alphas to be the rewiring probability (social selection vs influence)
    # Set gammas to be the transitivity parameter
    params = [[G], rhos, alphas]
    params = [p for p in product(*params)]

    # Get number of cpus that can be used
    num_cores = multiprocessing.cpu_count()

    # Actual call to parallel function
    results = Parallel(n_jobs=num_cores)(delayed(ParallelCall_RewireTriangle)(param) for param in tqdm(params))


In [ ]:
def triangle_rewire(H, E, N, Opinions, Count, DiscordantIndex, DiscordantEdges,
                    edge_id, edgeChoice, gamma):
    """
    Performs transitivity-enforcing rewiring by swapping nodes between edges,
    preferring swaps that enforce triangle closure with probability gamma,
    falling back to random rewiring otherwise.

    Input: H: Hypergraph dict
           E: List of edge ids
           N: Node neighborhood dict
           Opinions: Node opinion dict
           Count: Edge opinion-1 count dict
           DiscordantIndex: Reverse index for O(1) discordant edge removal
           DiscordantEdges: List of discordant edges
           edge_id: Selected edge to rewire
           edgeChoice: Index of edge in DiscordantEdges
           gamma: Probability of triangle rewiring
    """
    sourceEdge = H[edge_id]
    sourceNode_id = np.random.randint(len(sourceEdge))
    sourceNode = sourceEdge[sourceNode_id]

    roll = random.random()
    NofNs = [None]
    found = False
    targetEdge_id = None
    targetNode = None

    if roll < gamma:

        Neighbors = set(
            v for e in N[sourceNode] if e != edge_id for v in H[e] if v != sourceNode
        )
        NeighborEdges = set(
            e for v in Neighbors for e in N[v] if e not in N[sourceNode]
        )
        NofNs = list(set(
            v for e in NeighborEdges for v in H[e] if v not in Neighbors and v != sourceNode
        ))

        while NofNs:
            middleNode_id = np.random.randint(len(NofNs))
            middleNode = NofNs[middleNode_id]#random.choice(NofNs)
            temp = []
            counter = 0
            for e in N[middleNode]:
                if e in NeighborEdges:
                    counter += 1
                else:
                    temp.append(e)

            cands = list(N[middleNode]) if counter > 1 else temp

            while cands:
                cand_edge_id = np.random.randint(len(cands))
                cand_edge = H[cands[cand_edge_id]]
                valid_nodes = [v for v in cand_edge
                               if v != middleNode and v not in sourceEdge]
                if valid_nodes:
                    targetEdge_id = cands[cand_edge_id]
                    targetNode = random.choice(valid_nodes)
                    found = True
                    break
                else:
                    # swap-and-pop for O(1) removal
                    cands[cand_edge_id] = cands[-1]
                    cands.pop()

            if found:
                break

            # swap-and-pop for O(1) removal
            NofNs[middleNode_id] = NofNs[-1]
            NofNs.pop()

    # Fallback to random rewiring if gamma branch failed or was not taken
    if not found:
        while True:
            targetEdge_id = random.choice(E)
            if targetEdge_id == edge_id:
                continue
            targetEdge = H[targetEdge_id]
            targetNode_id = np.random.randint(len(targetEdge))
            targetNode = targetEdge[targetNode_id]
            if (targetNode != sourceNode and sourceNode not in targetEdge and targetNode not in sourceEdge):
                break

    # Update source edge
    new_sourceEdge = list(sourceEdge)
    new_sourceEdge[sourceNode_id] = targetNode
    H[edge_id] = sorted(new_sourceEdge)

    Count[edge_id] += Opinions[targetNode] - Opinions[sourceNode]
    if Count[edge_id] == 0 or Count[edge_id] == len(H[edge_id]):
        remove_discordant(edge_id, DiscordantIndex, DiscordantEdges)

    # Update target edge
    count = Count[targetEdge_id]
    new_targetEdge = list(H[targetEdge_id])
    targetNode_idx = new_targetEdge.index(targetNode)
    new_targetEdge[targetNode_idx] = sourceNode
    H[targetEdge_id] = sorted(new_targetEdge)
    Count[targetEdge_id] += Opinions[sourceNode] - Opinions[targetNode]

    if (count == 0 or count == len(H[targetEdge_id])) and (Count[targetEdge_id] != 0 and Count[targetEdge_id] != len(H[targetEdge_id])):
        add_discordant(targetEdge_id, DiscordantIndex, DiscordantEdges)
    elif (count != 0 and count != len(H[targetEdge_id])) and (Count[targetEdge_id] == 0 or Count[targetEdge_id] == len(H[targetEdge_id])):
        remove_discordant(targetEdge_id, DiscordantIndex, DiscordantEdges)

    # Update node neighborhoods
    N[sourceNode].remove(edge_id)
    N[sourceNode].add(targetEdge_id)
    N[targetNode].add(edge_id)
    N[targetNode].remove(targetEdge_id)

def HG_TriangleRewire_Voter(H, alpha, rho, gamma, voting='Majority',
                             exit_criteria=np.inf, timing=False):
    """
    Implementation of the hypergraph triangle-rewire model using
    a node-swapping rewiring rule.
    Input: Initial hypergraph dict H = {edge_id: edge_list}
           alpha: Probability of rewiring
           rho: Initial proportion of opinion 1 in H
           gamma: Probability of triangle rewiring
           voting: Rule used for voting, either 'Majority' or 'Proportional'
           exit_criteria: Maximum number of steps before halting
           timing: Whether to print timing information
    Output: List of proportions of opinion 1 at each time step, and
            terminal hypergraph H.
    """
    # Define vote_function once before loop — avoids repeated string comparison
    vote_function = majority_vote if voting == 'Majority' else proportional_vote

    if timing:
        print("Initializing hypergraph, variables and data structures", flush=True)
        start = time.time()

    V = set(v for e in H for v in H[e])
    E = list(H.keys())

    N = {v: set() for v in V}
    for e in H:
        for v in H[e]:
            N[v].add(e)

    Opinions = set(np.random.choice(list(V), size=round(rho * len(V)), replace=False))
    Opinions = {v: 1 if v in Opinions else 0 for v in V}

    # Use integer Count instead of float Prop — avoids floating point equality issues
    Count = {e: sum(Opinions[v] for v in H[e]) for e in H}

    # Use DiscordantIndex for O(1) add/remove — matches rewire_random pattern
    DiscordantEdges = [e for e in H if 0 < Count[e] < len(H[e])]
    DiscordantIndex = {edge_id: pos for pos, edge_id in enumerate(DiscordantEdges)}

    Proportions = [sum(Opinions.values()) / len(V)]
    DiscordantCounts = [len(DiscordantEdges) / len(E)]
    timer = 0

    if timing:
        end = time.time()
        print("Initialization complete, time taken: " + str(end - start) + " seconds",
              flush=True)
        print("Beginning network evolution", flush=True)
        start = time.time()

    while len(DiscordantEdges) > 0 and timer < exit_criteria:
        timer += 1
        Proportions.append(Proportions[-1])

        edgeChoice = np.random.randint(len(DiscordantEdges))
        edge_id = DiscordantEdges[edgeChoice]

        if random.random() < alpha:
            triangle_rewire(H, E, N, Opinions, Count, DiscordantIndex, DiscordantEdges,
                           edge_id, edgeChoice, gamma)

        else:
            num_changed, new_opinion = social_influence(
                H, N, Opinions, Count,
                DiscordantIndex, DiscordantEdges,
                edge_id, edgeChoice, vote_function
            )
            if new_opinion == 1:
                Proportions[timer] += num_changed / len(V)
            else:
                Proportions[timer] -= num_changed / len(V)

        DiscordantCounts.append(len(DiscordantEdges) / len(E))

    if timing:
        end = time.time()
        print("Evolution complete, time taken: " + str(end - start) + " seconds",
              flush=True)

    return np.array(Proportions), np.array(DiscordantCounts), H